In [ ]:
# Using this document for first iteration of testing for low resource + low resolution deepfake detection

In [ ]:
## FF++ with undersampling
!pip install tqdm scikit-learn matplotlib pillow opencv-python-headless thop pandas

import os, gc, cv2, json, time, random, copy, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    from thop import profile
    THOP_AVAILABLE = True
except Exception:
    THOP_AVAILABLE = False

# =========================
# Config
# =========================
FRAMES_DIR = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames"
METADATA_CSV = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv"

BALANCE_MODE = "undersample"
# options: "none", "undersample", "oversample"

RESULTS_DIR = f"./gated_research_results_ffpp_{BALANCE_MODE}"
os.makedirs(RESULTS_DIR, exist_ok=True)

RESOLUTIONS = [128, 224, 256, 384]
IMAGE_SIZE_FOR_MODEL = 224

BATCH_SIZE = 32
EPOCHS = 15
LR = 2e-4
WEIGHT_DECAY = 1e-4
SEED = 42
UNCERTAIN_THRESHOLD = 0.65
USE_AMP = True
USE_TORCH_COMPILE = False

CACHE_FEATURES = False
NUM_WORKERS = 2
PIN_MEMORY = True
PERSISTENT_WORKERS = False
PREFETCH_FACTOR = 2

BRANCH_CHANNELS = (12, 20, 32, 48)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_gated_dual_branch_ffpp.pth")
HISTORY_JSON = os.path.join(RESULTS_DIR, "history_ffpp.json")
SUMMARY_JSON = os.path.join(RESULTS_DIR, "summary_ffpp.json")
OBSERVATIONS_TXT = os.path.join(RESULTS_DIR, "observations_ffpp.txt")
CONFUSION_MATRIX_PNG = os.path.join(RESULTS_DIR, "confusion_matrix_test_gated_ffpp.png")
CURVES_PNG = os.path.join(RESULTS_DIR, "training_curves_ffpp.png")

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)
print("THOP available:", THOP_AVAILABLE)
print("Balance mode:", BALANCE_MODE)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def format_seconds(sec):
    return f"{sec:.2f} sec" if sec < 60 else f"{sec/60:.2f} min"

# =========================
# Metadata + balancing
# =========================
def load_ffpp_metadata(metadata_csv):
    df = pd.read_csv(metadata_csv)
    required_cols = {"frame_path", "split", "label", "label_name", "video_id"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"metadata.csv missing columns: {missing}")

    df = df[df["frame_path"].apply(os.path.exists)].reset_index(drop=True)
    df["is_oversampled_real"] = False

    print("\nLoaded FF++ metadata")
    print(f"Total usable frames: {len(df)}")
    return df

def scan_ffpp_dataset(metadata_csv):
    df = load_ffpp_metadata(metadata_csv)
    print("\nScanning FF++ dataset structure...")
    for split in ["Train", "Val", "Test"]:
        split_df = df[df["split"].str.lower() == split.lower()]
        real_count = int((split_df["label"] == 0).sum())
        fake_count = int((split_df["label"] == 1).sum())
        print(f"{split}: real={real_count}, fake={fake_count}, total={len(split_df)}")
    return df

def balance_train_metadata(train_df, mode="none", seed=42):
    train_df = train_df.copy().reset_index(drop=True)
    train_df["is_oversampled_real"] = False

    if mode == "none":
        print("\nUsing original imbalanced train set")
        print(train_df["label_name"].value_counts())
        return train_df

    real_df = train_df[train_df["label"] == 0].copy()
    fake_df = train_df[train_df["label"] == 1].copy()

    if mode == "undersample":
        n = min(len(real_df), len(fake_df))
        real_bal = real_df.sample(n=n, random_state=seed)
        fake_bal = fake_df.sample(n=n, random_state=seed)

        out = (
            pd.concat([real_bal, fake_bal])
            .sample(frac=1, random_state=seed)
            .reset_index(drop=True)
        )
        out["is_oversampled_real"] = False

        print("\nUsing undersampled train set")
        print(out["label_name"].value_counts())
        return out

    if mode == "oversample":
        real_os = real_df.sample(
            n=len(fake_df),
            replace=True,
            random_state=seed
        ).copy()

        real_os["is_oversampled_real"] = True
        fake_df["is_oversampled_real"] = False

        out = (
            pd.concat([real_os, fake_df])
            .sample(frac=1, random_state=seed)
            .reset_index(drop=True)
        )

        print("\nUsing oversampled train set")
        print(out["label_name"].value_counts())
        print("Oversampled real rows:", int(out["is_oversampled_real"].sum()))
        return out

    raise ValueError(f"Unknown BALANCE_MODE: {mode}")

def build_base_samples_from_metadata(df, split):
    if "split" in df.columns:
        split_df = df[df["split"].str.lower() == split.lower()].reset_index(drop=True)
    else:
        split_df = df.reset_index(drop=True)

    samples = []
    for _, row in split_df.iterrows():
        samples.append({
            "path": row["frame_path"],
            "label": int(row["label"]),
            "split": row.get("split", split),
            "video_id": row["video_id"],
            "is_oversampled_real": bool(row.get("is_oversampled_real", False)),
        })
    return samples

def print_dataset_sanity(train_samples, val_samples, test_samples):
    print("\nDataset sanity check")
    print(f"Train base images: {len(train_samples)}")
    print(f"Val base images:   {len(val_samples)}")
    print(f"Test base images:  {len(test_samples)}")
    print(f"Train effective samples per epoch: {len(train_samples)}")
    print(f"Val effective samples:             {len(val_samples) * len(RESOLUTIONS)}")
    print(f"Test effective samples:            {len(test_samples) * len(RESOLUTIONS)}")

    train_counts = Counter(s["label"] for s in train_samples)
    val_counts = Counter(s["label"] for s in val_samples)
    test_counts = Counter(s["label"] for s in test_samples)

    print("\nBase image class counts:")
    print(f"  Train: real={train_counts[0]}, fake={train_counts[1]}")
    print(f"  Val:   real={val_counts[0]}, fake={val_counts[1]}")
    print(f"  Test:  real={test_counts[0]}, fake={test_counts[1]}")

# =========================
# Quality features and DCT
# =========================
def compute_blur_and_sharpness_from_rgb(np_rgb):
    gray = cv2.cvtColor(np_rgb, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    sharpness = float(np.var(lap))
    blur_proxy = 1.0 / (sharpness + 1e-6)
    return blur_proxy, sharpness

def compute_jpeg_compression_proxy(path):
    file_size_kb = os.path.getsize(path) / 1024.0
    ext = Path(path).suffix.lower()
    return float(1.0 / (file_size_kb + 1e-6)) if ext in [".jpg", ".jpeg"] else 0.0

def compute_dct_tensor(pil_img, out_size=224):
    gray = np.array(pil_img.convert("L")).astype(np.float32) / 255.0
    dct = cv2.dct(gray)
    dct = np.log1p(np.abs(dct))
    dct = (dct - dct.min()) / (dct.max() - dct.min() + 1e-6)
    dct = cv2.resize(dct, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    return torch.tensor(dct, dtype=torch.float32).unsqueeze(0)

def build_quality_features(pil_img, path, resized_resolution):
    np_rgb = np.array(pil_img.convert("RGB"))
    blur_proxy, sharpness = compute_blur_and_sharpness_from_rgb(np_rgb)
    compression_proxy = compute_jpeg_compression_proxy(path)

    feat = np.array(
        [
            resized_resolution / 224.0,
            min(blur_proxy * 100.0, 10.0),
            min(sharpness / 500.0, 10.0),
            min(compression_proxy * 50.0, 10.0),
            np.mean(np_rgb) / 255.0,
            np.std(np_rgb) / 255.0,
        ],
        dtype=np.float32,
    )
    return feat

def make_gate_targets_from_quality_features(qf_batch):
    targets = []
    qf_np = qf_batch.detach().cpu().numpy()

    for qf in qf_np:
        res_norm, blur_norm, sharp_norm, compression_norm, brightness, contrast = qf
        approx_res = res_norm * 224.0

        if approx_res >= 160 and sharp_norm >= 0.18 and compression_norm <= 0.8:
            targets.append(0)
        elif approx_res >= 128 and sharp_norm >= 0.06:
            targets.append(1)
        else:
            targets.append(2)

    return torch.tensor(targets, dtype=torch.long)

# =========================
# Dataset
# =========================
class MultiResFFPPDataset(Dataset):
    def __init__(
        self,
        base_samples,
        image_size=224,
        resolutions=(128, 224, 256, 384),
        mode="train_random",
        cache_features=False
    ):
        self.base_samples = base_samples
        self.image_size = image_size
        self.resolutions = list(resolutions)
        self.mode = mode
        self.cache_features = cache_features
        self.cache = {}

        if self.mode == "eval_all":
            self.samples = []
            for s in self.base_samples:
                for res in self.resolutions:
                    self.samples.append({**s, "resolution": res})
        else:
            self.samples = self.base_samples

        self.mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)

    def __len__(self):
        return len(self.samples)

    def pil_to_rgb_tensor(self, pil_img):
        arr = np.array(pil_img).astype(np.float32) / 255.0
        arr = np.transpose(arr, (2, 0, 1))
        tensor = torch.from_numpy(arr).float()
        return (tensor - self.mean) / self.std

    def _augment_oversampled_real(self, img):
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() < 0.35:
            angle = random.uniform(-12, 12)
            img = img.rotate(angle)
        if random.random() < 0.35:
            factor = random.uniform(0.85, 1.15)
            arr = np.array(img).astype(np.float32)
            arr = np.clip(arr * factor, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr)
        return img

    def _make_item(self, path, label, resolution, is_oversampled_real=False):
        cache_key = (path, resolution, is_oversampled_real)

        if self.cache_features and cache_key in self.cache:
            rgb, freq, quality_feat = self.cache[cache_key]
        else:
            with Image.open(path) as img:
                img = img.convert("RGB")
                img_res = img.resize((resolution, resolution), Image.BILINEAR)

            if self.mode == "train_random" and is_oversampled_real:
                img_res = self._augment_oversampled_real(img_res)

            quality_feat_np = build_quality_features(img_res, path, resolution)
            img_for_model = img_res.resize((self.image_size, self.image_size), Image.BILINEAR)

            rgb = self.pil_to_rgb_tensor(img_for_model)
            freq = compute_dct_tensor(img_for_model, out_size=self.image_size)
            quality_feat = torch.tensor(quality_feat_np, dtype=torch.float32)

            if self.cache_features:
                self.cache[cache_key] = (rgb, freq, quality_feat)

        return {
            "rgb": rgb,
            "freq": freq,
            "quality_features": quality_feat,
            "label": torch.tensor(label, dtype=torch.long),
            "resolution": torch.tensor(resolution, dtype=torch.long),
            "path": path,
        }

    def __getitem__(self, idx):
        sample = self.samples[idx]
        path = sample["path"]
        label = sample["label"]
        is_oversampled_real = sample.get("is_oversampled_real", False)

        if self.mode == "train_random":
            resolution = random.choice(self.resolutions)
        else:
            resolution = sample["resolution"]

        return self._make_item(path, label, resolution, is_oversampled_real)

# =========================
# Model blocks
# =========================
class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=None, groups=1, act=True):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True) if act else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = ConvBNAct(in_ch, in_ch, kernel_size=3, stride=stride, groups=in_ch)
        self.pw = ConvBNAct(in_ch, out_ch, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        return self.pw(self.dw(x))

class ExitHead(nn.Module):
    def __init__(self, in_ch, hidden_dim=64, num_classes=2, dropout=0.2):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_ch, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        feat = self.pool(x).flatten(1)
        logits = self.fc(feat)
        return logits, feat

class AdaptiveGate(nn.Module):
    def __init__(self, in_features=6, hidden_dim=32, num_routes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, num_routes),
        )

    def forward(self, qf):
        logits = self.net(qf)
        probs = torch.softmax(logits, dim=1)
        route_idx = probs.argmax(dim=1)
        return logits, probs, route_idx

class EarlyExitBranch(nn.Module):
    def __init__(self, in_ch, channels=(12, 20, 32, 48), num_classes=2, exit_hidden=48):
        super().__init__()
        c1, c2, c3, c4 = channels

        self.stem = ConvBNAct(in_ch, c1, kernel_size=3, stride=2)
        self.stage1 = nn.Sequential(
            DepthwiseSeparableBlock(c1, c2, stride=2),
            DepthwiseSeparableBlock(c2, c2, stride=1),
        )
        self.stage2 = nn.Sequential(
            DepthwiseSeparableBlock(c2, c3, stride=2),
            DepthwiseSeparableBlock(c3, c3, stride=1),
        )
        self.stage3 = nn.Sequential(
            DepthwiseSeparableBlock(c3, c4, stride=2),
            DepthwiseSeparableBlock(c4, c4, stride=1),
        )

        self.exit1 = ExitHead(c2, hidden_dim=exit_hidden, num_classes=num_classes)
        self.exit2 = ExitHead(c3, hidden_dim=exit_hidden, num_classes=num_classes)
        self.exit3 = ExitHead(c4, hidden_dim=exit_hidden, num_classes=num_classes)

    def forward_all(self, x):
        x = self.stem(x)

        x1 = self.stage1(x)
        e1_logits, e1_feat = self.exit1(x1)

        x2 = self.stage2(x1)
        e2_logits, e2_feat = self.exit2(x2)

        x3 = self.stage3(x2)
        e3_logits, e3_feat = self.exit3(x3)

        return {
            "exit1_logits": e1_logits,
            "exit2_logits": e2_logits,
            "exit3_logits": e3_logits,
            "deep_feat": e3_feat,
        }

    def forward_route(self, x, route):
        x = self.stem(x)
        x1 = self.stage1(x)

        if route == 0:
            logits, feat = self.exit1(x1)
            return logits, feat

        x2 = self.stage2(x1)

        if route == 1:
            logits, feat = self.exit2(x2)
            return logits, feat

        x3 = self.stage3(x2)
        logits, feat = self.exit3(x3)
        return logits, feat

class GatedDualBranchDeepfakeNet(nn.Module):
    def __init__(self, branch_channels=(12, 20, 32, 48), gate_features=6, num_classes=2):
        super().__init__()
        self.spatial = EarlyExitBranch(3, channels=branch_channels, num_classes=num_classes, exit_hidden=48)
        self.frequency = EarlyExitBranch(1, channels=branch_channels, num_classes=num_classes, exit_hidden=48)
        self.gate = AdaptiveGate(in_features=gate_features, hidden_dim=32, num_routes=3)

        deep_dim = branch_channels[-1]
        self.fusion = nn.Sequential(
            nn.Linear(deep_dim * 2, 96),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(96, num_classes),
        )

    def forward(self, rgb, freq, qf):
        gate_logits, gate_probs, gate_routes = self.gate(qf)

        spatial_out = self.spatial.forward_all(rgb)
        freq_out = self.frequency.forward_all(freq)

        fused_feat = torch.cat([spatial_out["deep_feat"], freq_out["deep_feat"]], dim=1)
        fused_logits = self.fusion(fused_feat)

        return {
            "gate_logits": gate_logits,
            "gate_probs": gate_probs,
            "route_idx": gate_routes,
            "spatial_exit1": spatial_out["exit1_logits"],
            "spatial_exit2": spatial_out["exit2_logits"],
            "spatial_exit3": spatial_out["exit3_logits"],
            "freq_exit1": freq_out["exit1_logits"],
            "freq_exit2": freq_out["exit2_logits"],
            "freq_exit3": freq_out["exit3_logits"],
            "fused_logits": fused_logits,
        }

    def predict_fast_compute(self, rgb, freq, qf, threshold=0.65, forced_route=None):
        if forced_route is None:
            _, _, routes = self.gate(qf)
            if len(torch.unique(routes)) > 1:
                return self.predict(rgb, freq, qf, threshold=threshold, forced_route=None)
            route = int(routes[0].item())
        else:
            route = int(forced_route)
            routes = torch.full((rgb.size(0),), route, dtype=torch.long, device=rgb.device)

        if route in [0, 1]:
            s_logits, _ = self.spatial.forward_route(rgb, route)
            f_logits, _ = self.frequency.forward_route(freq, route)
            final_logits = 0.5 * s_logits + 0.5 * f_logits
        else:
            _, s_feat = self.spatial.forward_route(rgb, 2)
            _, f_feat = self.frequency.forward_route(freq, 2)
            final_logits = self.fusion(torch.cat([s_feat, f_feat], dim=1))

        probs = torch.softmax(final_logits, dim=1)
        conf, pred = probs.max(dim=1)

        decision = []
        for c, p in zip(conf.tolist(), pred.tolist()):
            if c < threshold:
                decision.append("uncertain")
            elif p == 0:
                decision.append("real")
            else:
                decision.append("deepfake")

        return {
            "logits": final_logits,
            "probs": probs,
            "pred": pred,
            "conf": conf,
            "decision": decision,
            "routes": routes,
        }

    def predict(self, rgb, freq, qf, threshold=0.65, forced_route=None):
        out = self.forward(rgb, freq, qf)
        B = rgb.size(0)

        routes = out["route_idx"] if forced_route is None else torch.full((B,), forced_route, dtype=torch.long, device=rgb.device)
        final_logits = []

        for i in range(B):
            route = routes[i].item()
            if route == 0:
                logits_i = 0.5 * out["spatial_exit1"][i:i+1] + 0.5 * out["freq_exit1"][i:i+1]
            elif route == 1:
                logits_i = 0.5 * out["spatial_exit2"][i:i+1] + 0.5 * out["freq_exit2"][i:i+1]
            else:
                logits_i = out["fused_logits"][i:i+1]
            final_logits.append(logits_i)

        final_logits = torch.cat(final_logits, dim=0)
        probs = torch.softmax(final_logits, dim=1)
        conf, pred = probs.max(dim=1)

        decision = []
        for c, p in zip(conf.tolist(), pred.tolist()):
            if c < threshold:
                decision.append("uncertain")
            elif p == 0:
                decision.append("real")
            else:
                decision.append("deepfake")

        return {
            "logits": final_logits,
            "probs": probs,
            "pred": pred,
            "conf": conf,
            "decision": decision,
            "routes": routes,
            "raw_outputs": out,
        }

# =========================
# Loss and metrics
# =========================
def compute_total_loss(outputs, labels, gate_targets, gate_weight=0.20):
    loss = 0.0
    loss += 0.08 * F.cross_entropy(outputs["spatial_exit1"], labels)
    loss += 0.12 * F.cross_entropy(outputs["spatial_exit2"], labels)
    loss += 0.15 * F.cross_entropy(outputs["spatial_exit3"], labels)

    loss += 0.08 * F.cross_entropy(outputs["freq_exit1"], labels)
    loss += 0.12 * F.cross_entropy(outputs["freq_exit2"], labels)
    loss += 0.15 * F.cross_entropy(outputs["freq_exit3"], labels)

    loss += 0.30 * F.cross_entropy(outputs["fused_logits"], labels)
    loss += gate_weight * F.cross_entropy(outputs["gate_logits"], gate_targets)
    return loss

def compute_classification_metrics(y_true, y_pred, y_score):
    acc = accuracy_score(y_true, y_pred)

    try:
        auc = roc_auc_score(y_true, y_score)
    except Exception:
        auc = 0.0

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)

    return {
        "accuracy": float(acc),
        "auc": float(auc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "confusion_matrix": cm.tolist(),
    }

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

class RouteWrapper(nn.Module):
    def __init__(self, model, route):
        super().__init__()
        self.model = model
        self.route = route

    def forward(self, rgb, freq, qf):
        pred = self.model.predict_fast_compute(rgb, freq, qf, threshold=0.0, forced_route=self.route)
        return pred["logits"]

def remove_thop_buffers(module):
    for m in module.modules():
        for attr in ["total_ops", "total_params"]:
            if hasattr(m, attr):
                try:
                    delattr(m, attr)
                except Exception:
                    pass

def measure_flops(model, image_size=224, device="cpu"):
    if not THOP_AVAILABLE:
        return None

    rgb = torch.randn(1, 3, image_size, image_size).to(device)
    freq = torch.randn(1, 1, image_size, image_size).to(device)
    qf = torch.randn(1, 6).to(device)

    flops_info = {}
    for name, route in {"fast_route": 0, "medium_route": 1, "full_route": 2}.items():
        wrapper = RouteWrapper(copy.deepcopy(model), route).to(device).eval()
        remove_thop_buffers(wrapper)

        try:
            macs, params = profile(wrapper, inputs=(rgb, freq, qf), verbose=False)
            flops_info[name] = {
                "macs": float(macs),
                "flops": float(2 * macs),
                "params": float(params),
            }
        except Exception as e:
            print(f"THOP profiling failed for {name}: {e}")
            flops_info[name] = None
        finally:
            remove_thop_buffers(wrapper)
            del wrapper
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()

    return flops_info

def move_batch_to_device(batch, device, use_channels_last=False):
    rgb = batch["rgb"].to(device, non_blocking=True)
    freq = batch["freq"].to(device, non_blocking=True)
    qf = batch["quality_features"].to(device, non_blocking=True)
    labels = batch["label"].to(device, non_blocking=True)

    if use_channels_last:
        rgb = rgb.contiguous(memory_format=torch.channels_last)
        freq = freq.contiguous(memory_format=torch.channels_last)

    return rgb, freq, qf, labels

# =========================
# Train and eval
# =========================
def train_one_epoch(model, loader, optimizer, scaler, device, epoch_idx):
    model.train()

    total_loss = 0.0
    total_count = 0
    running_y_true, running_y_pred, running_y_score = [], [], []

    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx+1}", leave=False)

    for batch in pbar:
        rgb, freq, qf, labels = move_batch_to_device(
            batch, device, use_channels_last=(device == "cuda")
        )

        gate_targets = make_gate_targets_from_quality_features(qf).to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(USE_AMP and device == "cuda")):
            outputs = model(rgb, freq, qf)
            loss = compute_total_loss(outputs, labels, gate_targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            probs = torch.softmax(outputs["fused_logits"], dim=1)[:, 1]
            preds = outputs["fused_logits"].argmax(dim=1)

        bs = rgb.size(0)
        total_loss += loss.item() * bs
        total_count += bs

        running_y_true.extend(labels.detach().cpu().numpy().tolist())
        running_y_pred.extend(preds.detach().cpu().numpy().tolist())
        running_y_score.extend(probs.detach().cpu().numpy().tolist())

        pbar.set_postfix({
            "loss": f"{total_loss / max(total_count, 1):.4f}",
            "acc": f"{accuracy_score(running_y_true, running_y_pred):.4f}",
        })

    metrics = compute_classification_metrics(running_y_true, running_y_pred, running_y_score)
    metrics["loss"] = float(total_loss / max(total_count, 1))
    return metrics

@torch.no_grad()
def evaluate_model(model, loader, device, split_name="Val", forced_route=None, threshold=0.65, fast_compute=False):
    model.eval()

    all_y_true, all_y_pred, all_y_score = [], [], []
    all_decisions, all_routes, all_resolutions = [], [], []

    start_time = time.time()
    pbar = tqdm(loader, desc=f"Eval {split_name}", leave=False)

    for batch in pbar:
        rgb = batch["rgb"].to(device, non_blocking=True)
        freq = batch["freq"].to(device, non_blocking=True)
        qf = batch["quality_features"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        resolutions = batch["resolution"].cpu().numpy().tolist()

        if device == "cuda":
            rgb = rgb.contiguous(memory_format=torch.channels_last)
            freq = freq.contiguous(memory_format=torch.channels_last)

        if fast_compute and forced_route is not None:
            pred_out = model.predict_fast_compute(rgb, freq, qf, threshold=threshold, forced_route=forced_route)
        else:
            pred_out = model.predict(rgb, freq, qf, threshold=threshold, forced_route=forced_route)

        all_y_true.extend(labels.detach().cpu().numpy().tolist())
        all_y_pred.extend(pred_out["pred"].detach().cpu().numpy().tolist())
        all_y_score.extend(pred_out["probs"][:, 1].detach().cpu().numpy().tolist())
        all_decisions.extend(pred_out["decision"])
        all_routes.extend(pred_out["routes"].detach().cpu().numpy().tolist())
        all_resolutions.extend(resolutions)

    elapsed = time.time() - start_time
    metrics = compute_classification_metrics(all_y_true, all_y_pred, all_y_score)

    route_counter = Counter(all_routes)
    metrics.update({
        "num_samples": int(len(all_y_true)),
        "uncertain_rate_pct": float(100.0 * sum(d == "uncertain" for d in all_decisions) / max(len(all_decisions), 1)),
        "route_usage": {
            "fast_exit1_pct": 100.0 * route_counter.get(0, 0) / max(len(all_routes), 1),
            "medium_exit2_pct": 100.0 * route_counter.get(1, 0) / max(len(all_routes), 1),
            "full_exit3_pct": 100.0 * route_counter.get(2, 0) / max(len(all_routes), 1),
        },
        "elapsed_sec": float(elapsed),
        "elapsed_pretty": format_seconds(elapsed),
    })

    per_resolution = {}
    for res in sorted(set(all_resolutions)):
        idxs = [i for i, r in enumerate(all_resolutions) if r == res]
        y_true_res = [all_y_true[i] for i in idxs]
        y_pred_res = [all_y_pred[i] for i in idxs]
        y_score_res = [all_y_score[i] for i in idxs]
        decisions_res = [all_decisions[i] for i in idxs]
        routes_res = [all_routes[i] for i in idxs]

        res_metrics = compute_classification_metrics(y_true_res, y_pred_res, y_score_res)
        rc = Counter(routes_res)

        res_metrics["uncertain_rate_pct"] = float(100.0 * sum(d == "uncertain" for d in decisions_res) / max(len(decisions_res), 1))
        res_metrics["route_usage"] = {
            "fast_exit1_pct": 100.0 * rc.get(0, 0) / max(len(routes_res), 1),
            "medium_exit2_pct": 100.0 * rc.get(1, 0) / max(len(routes_res), 1),
            "full_exit3_pct": 100.0 * rc.get(2, 0) / max(len(routes_res), 1),
        }
        res_metrics["num_samples"] = int(len(idxs))
        per_resolution[str(res)] = res_metrics

    metrics["per_resolution"] = per_resolution
    return metrics

def print_eval_summary(name, metrics):
    print(f"\n{name}")
    print(f"  Accuracy:        {metrics['accuracy']:.4f}")
    print(f"  AUC:             {metrics['auc']:.4f}")
    print(f"  Precision:       {metrics['precision']:.4f}")
    print(f"  Recall:          {metrics['recall']:.4f}")
    print(f"  F1:              {metrics['f1']:.4f}")
    print(f"  Uncertain rate:  {metrics['uncertain_rate_pct']:.2f}%")
    print(
        f"  Route usage: fast={metrics['route_usage']['fast_exit1_pct']:.2f}% "
        f"medium={metrics['route_usage']['medium_exit2_pct']:.2f}% "
        f"full={metrics['route_usage']['full_exit3_pct']:.2f}%"
    )

def save_confusion_matrix(cm, path, title="Confusion Matrix"):
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["real", "fake"])
    plt.yticks(tick_marks, ["real", "fake"])
    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i][j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()

def plot_training_curves(history, path):
    epochs = list(range(1, len(history["train_loss"]) + 1))
    plt.figure(figsize=(14, 4))

    plt.subplot(1, 3, 1)
    plt.plot(epochs, history["train_loss"], marker="o", label="train_loss")
    plt.plot(epochs, history["val_loss"], marker="o", label="val_loss")
    plt.legend()
    plt.title("Loss")
    plt.xlabel("Epoch")

    plt.subplot(1, 3, 2)
    plt.plot(epochs, history["train_acc"], marker="o", label="train_acc")
    plt.plot(epochs, history["val_acc"], marker="o", label="val_acc")
    plt.legend()
    plt.title("Accuracy")
    plt.xlabel("Epoch")

    plt.subplot(1, 3, 3)
    plt.plot(epochs, history["val_auc"], marker="o", label="val_auc")
    plt.legend()
    plt.title("Validation AUC")
    plt.xlabel("Epoch")

    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()

@torch.no_grad()
def measure_inference_time(model, loader, device, forced_route=None, threshold=0.65, num_batches=20, fast_compute=True):
    model.eval()
    timings = []
    seen = 0

    for batch in loader:
        rgb = batch["rgb"].to(device, non_blocking=True)
        freq = batch["freq"].to(device, non_blocking=True)
        qf = batch["quality_features"].to(device, non_blocking=True)

        if device == "cuda":
            rgb = rgb.contiguous(memory_format=torch.channels_last)
            freq = freq.contiguous(memory_format=torch.channels_last)
            torch.cuda.synchronize()

        start = time.time()

        if fast_compute and forced_route is not None:
            _ = model.predict_fast_compute(rgb, freq, qf, threshold=threshold, forced_route=forced_route)
        else:
            _ = model.predict(rgb, freq, qf, threshold=threshold, forced_route=forced_route)

        if device == "cuda":
            torch.cuda.synchronize()

        timings.append(time.time() - start)
        seen += 1
        if seen >= num_batches:
            break

    avg_batch = float(np.mean(timings))
    avg_img = avg_batch / loader.batch_size
    fps = 1.0 / avg_img if avg_img > 0 else 0.0

    return {
        "avg_batch_time_sec": avg_batch,
        "avg_image_time_sec": float(avg_img),
        "avg_image_time_ms": float(avg_img * 1000.0),
        "fps": float(fps),
    }

def generate_observations(summary):
    tg = summary["test_gated"]
    tf = summary["test_fast"]
    tm = summary["test_medium"]
    tfull = summary["test_full"]

    lines = []
    lines.append("AUTOMATIC OBSERVATIONS")
    lines.append("=" * 60)
    lines.append(f"Gated test accuracy: {tg['accuracy']:.4f}")
    lines.append(f"Fast-only test accuracy: {tf['accuracy']:.4f}")
    lines.append(f"Medium-only test accuracy: {tm['accuracy']:.4f}")
    lines.append(f"Full-only test accuracy: {tfull['accuracy']:.4f}")
    lines.append("")
    lines.append("Route behavior")
    lines.append(
        f"Gate used Exit1 for {tg['route_usage']['fast_exit1_pct']:.2f}%, "
        f"Exit2 for {tg['route_usage']['medium_exit2_pct']:.2f}%, "
        f"and Exit3 for {tg['route_usage']['full_exit3_pct']:.2f}%."
    )
    return "\n".join(lines)

# =========================
# Main
# =========================
print("\n" + "=" * 70)
print("GATED DUAL BRANCH DEEPFAKE DETECTOR - FF++")
print("=" * 70)

df_meta = scan_ffpp_dataset(METADATA_CSV)

train_df_full = df_meta[df_meta["split"].str.lower() == "train"].reset_index(drop=True)
val_df = df_meta[df_meta["split"].str.lower() == "val"].reset_index(drop=True)
test_df = df_meta[df_meta["split"].str.lower() == "test"].reset_index(drop=True)

train_df_used = balance_train_metadata(train_df_full, mode=BALANCE_MODE, seed=SEED)

train_samples = build_base_samples_from_metadata(train_df_used, "Train")
val_samples = build_base_samples_from_metadata(val_df, "Val")
test_samples = build_base_samples_from_metadata(test_df, "Test")

print_dataset_sanity(train_samples, val_samples, test_samples)

train_ds = MultiResFFPPDataset(
    train_samples,
    image_size=IMAGE_SIZE_FOR_MODEL,
    resolutions=RESOLUTIONS,
    mode="train_random",
    cache_features=CACHE_FEATURES,
)

val_ds = MultiResFFPPDataset(
    val_samples,
    image_size=IMAGE_SIZE_FOR_MODEL,
    resolutions=RESOLUTIONS,
    mode="eval_all",
    cache_features=CACHE_FEATURES,
)

test_ds = MultiResFFPPDataset(
    test_samples,
    image_size=IMAGE_SIZE_FOR_MODEL,
    resolutions=RESOLUTIONS,
    mode="eval_all",
    cache_features=CACHE_FEATURES,
)

sample = train_ds[0]
print("\nTensor sanity")
print("RGB tensor shape:", tuple(sample["rgb"].shape))
print("Frequency tensor shape:", tuple(sample["freq"].shape))
print("Quality feature shape:", tuple(sample["quality_features"].shape))
print("Label:", sample["label"].item())
print("Resolution bucket:", sample["resolution"].item())

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
    loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_ds, shuffle=False, drop_last=False, **loader_kwargs)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = GatedDualBranchDeepfakeNet(
    branch_channels=BRANCH_CHANNELS,
    gate_features=6,
    num_classes=2,
).to(DEVICE)

if DEVICE == "cuda":
    model = model.to(memory_format=torch.channels_last)

print("\nMODEL DEBUG")
print("model type:", type(model))
print("compiled?", "OptimizedModule" in str(type(model)) or hasattr(model, "_orig_mod"))

total_params, trainable_params = count_parameters(model)
print("\nModel summary")
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

flops_info = measure_flops(model, IMAGE_SIZE_FOR_MODEL, DEVICE)
if flops_info is not None:
    print("\nApprox FLOPs")
    for k, v in flops_info.items():
        if v is None:
            print(f"{k}: profiling failed")
        else:
            print(f"{k}: GFLOPs={v['flops'] / 1e9:.4f}, MACs={v['macs'] / 1e9:.4f}G")

print("\nTorch compile disabled. Cache disabled. Safer DataLoader active.")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE == "cuda"))

history = {
    "train_loss": [],
    "train_acc": [],
    "train_auc": [],
    "val_loss": [],
    "val_acc": [],
    "val_auc": [],
    "epoch_time_sec": [],
}

best_val_auc = -1.0
best_val_loss_proxy = float("inf")
total_train_start = time.time()

print("\nStarting training...")
for epoch in range(EPOCHS):
    epoch_start = time.time()
    print(f"\n{'='*25} Epoch {epoch+1}/{EPOCHS} {'='*25}")

    train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, DEVICE, epoch)
    val_metrics = evaluate_model(
        model,
        val_loader,
        DEVICE,
        split_name="Val",
        forced_route=None,
        threshold=UNCERTAIN_THRESHOLD,
    )

    epoch_time = time.time() - epoch_start

    history["train_loss"].append(train_metrics["loss"])
    history["train_acc"].append(train_metrics["accuracy"])
    history["train_auc"].append(train_metrics["auc"])
    history["val_loss"].append(1.0 - val_metrics["auc"])
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_auc"].append(val_metrics["auc"])
    history["epoch_time_sec"].append(epoch_time)

    print("\nTrain")
    print(f"  Loss:      {train_metrics['loss']:.4f}")
    print(f"  Accuracy:  {train_metrics['accuracy']:.4f}")
    print(f"  AUC:       {train_metrics['auc']:.4f}")

    print_eval_summary("Validation", val_metrics)
    print(f"  Epoch time: {format_seconds(epoch_time)}")

    current_val_loss_proxy = 1.0 - val_metrics["auc"]
    if (val_metrics["auc"] > best_val_auc) or (
        abs(val_metrics["auc"] - best_val_auc) < 1e-8 and current_val_loss_proxy < best_val_loss_proxy
    ):
        best_val_auc = val_metrics["auc"]
        best_val_loss_proxy = current_val_loss_proxy
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Saved new best checkpoint.")

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

total_train_time = time.time() - total_train_start
print(f"\nTotal training time: {format_seconds(total_train_time)}")

plot_training_curves(history, CURVES_PNG)
with open(HISTORY_JSON, "w") as f:
    json.dump(history, f, indent=2)

print("\nLoading best checkpoint...")
state_dict = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)

print("\nRunning final evaluations...")
test_gated = evaluate_model(model, test_loader, DEVICE, "Test Gated", forced_route=None, threshold=UNCERTAIN_THRESHOLD)
test_fast = evaluate_model(model, test_loader, DEVICE, "Test Fast", forced_route=0, threshold=UNCERTAIN_THRESHOLD, fast_compute=True)
test_medium = evaluate_model(model, test_loader, DEVICE, "Test Medium", forced_route=1, threshold=UNCERTAIN_THRESHOLD, fast_compute=True)
test_full = evaluate_model(model, test_loader, DEVICE, "Test Full", forced_route=2, threshold=UNCERTAIN_THRESHOLD, fast_compute=True)

print_eval_summary("TEST GATED", test_gated)
print_eval_summary("TEST FAST ONLY", test_fast)
print_eval_summary("TEST MEDIUM ONLY", test_medium)
print_eval_summary("TEST FULL ONLY", test_full)

save_confusion_matrix(test_gated["confusion_matrix"], CONFUSION_MATRIX_PNG, title="Test Confusion Matrix - Gated")

inference_benchmarks = {
    "gated": measure_inference_time(model, test_loader, DEVICE, forced_route=None, threshold=UNCERTAIN_THRESHOLD, num_batches=20, fast_compute=False),
    "fast": measure_inference_time(model, test_loader, DEVICE, forced_route=0, threshold=UNCERTAIN_THRESHOLD, num_batches=20, fast_compute=True),
    "medium": measure_inference_time(model, test_loader, DEVICE, forced_route=1, threshold=UNCERTAIN_THRESHOLD, num_batches=20, fast_compute=True),
    "full": measure_inference_time(model, test_loader, DEVICE, forced_route=2, threshold=UNCERTAIN_THRESHOLD, num_batches=20, fast_compute=True),
}

print("\nInference benchmarks")
for name, bench in inference_benchmarks.items():
    print(f"  {name}: {bench['avg_image_time_ms']:.2f} ms/image, FPS={bench['fps']:.2f}")

checkpoint_size_mb = os.path.getsize(BEST_MODEL_PATH) / (1024 * 1024)

summary = {
    "config": {
        "frames_dir": FRAMES_DIR,
        "metadata_csv": METADATA_CSV,
        "dataset_name": "FaceForensics++ C23",
        "balance_mode": BALANCE_MODE,
        "train_samples_original": int(len(train_df_full)),
        "train_samples_used": int(len(train_df_used)),
        "train_real_used": int((train_df_used["label"] == 0).sum()),
        "train_fake_used": int((train_df_used["label"] == 1).sum()),
        "val_samples": int(len(val_df)),
        "test_samples": int(len(test_df)),
        "val_real": int((val_df["label"] == 0).sum()),
        "val_fake": int((val_df["label"] == 1).sum()),
        "test_real": int((test_df["label"] == 0).sum()),
        "test_fake": int((test_df["label"] == 1).sum()),
        "resolutions": RESOLUTIONS,
        "image_size_for_model": IMAGE_SIZE_FOR_MODEL,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "uncertain_threshold": UNCERTAIN_THRESHOLD,
        "device": DEVICE,
        "num_workers": NUM_WORKERS,
        "cache_features": CACHE_FEATURES,
        "branch_channels": BRANCH_CHANNELS,
        "use_torch_compile": USE_TORCH_COMPILE,
        "train_mode": "one random source resolution per image per epoch",
        "eval_mode": "all requested source resolutions per image",
    },
    "model_stats": {
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "checkpoint_size_mb": float(checkpoint_size_mb),
        "flops_info": flops_info,
    },
    "timing": {
        "total_training_time_sec": float(total_train_time),
        "total_training_time_pretty": format_seconds(total_train_time),
    },
    "test_gated": test_gated,
    "test_fast": test_fast,
    "test_medium": test_medium,
    "test_full": test_full,
    "inference_benchmarks": inference_benchmarks,
}

with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

observations = generate_observations(summary)
with open(OBSERVATIONS_TXT, "w") as f:
    f.write(observations)

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"Balance mode:             {BALANCE_MODE}")
print(f"Best checkpoint:          {BEST_MODEL_PATH}")
print(f"Checkpoint size:          {checkpoint_size_mb:.2f} MB")
print(f"Total params:             {total_params:,}")
print(f"Trainable params:         {trainable_params:,}")
print(f"Total training time:      {format_seconds(total_train_time)}")
print(f"Training curves saved:    {CURVES_PNG}")
print(f"Confusion matrix saved:   {CONFUSION_MATRIX_PNG}")
print(f"History JSON saved:       {HISTORY_JSON}")
print(f"Summary JSON saved:       {SUMMARY_JSON}")
print(f"Observations saved:       {OBSERVATIONS_TXT}")

print("\nKey test results")
print(f"Gated Accuracy:           {test_gated['accuracy']:.4f}")
print(f"Gated AUC:                {test_gated['auc']:.4f}")
print(f"Gated Uncertain rate:     {test_gated['uncertain_rate_pct']:.2f}%")
print(f"Fast route usage:         {test_gated['route_usage']['fast_exit1_pct']:.2f}%")
print(f"Medium route usage:       {test_gated['route_usage']['medium_exit2_pct']:.2f}%")
print(f"Full route usage:         {test_gated['route_usage']['full_exit3_pct']:.2f}%")

print("\nObservations preview")
print("-" * 70)
print(observations)
print("-" * 70)

Torch version: 2.11.0+cu130
CUDA available: True
Device: cuda
THOP available: True
Balance mode: undersample

GATED DUAL BRANCH DEEPFAKE DETECTOR - FF++

Loaded FF++ metadata
Total usable frames: 55953

Scanning FF++ dataset structure...
Train: real=6400, fake=38356, total=44756
Val: real=800, fake=4800, total=5600
Test: real=800, fake=4797, total=5597

Using undersampled train set
label_name
Real    6400
Fake    6400
Name: count, dtype: int64

Dataset sanity check
Train base images: 12800
Val base images:   5600
Test base images:  5597
Train effective samples per epoch: 12800
Val effective samples:             22400
Test effective samples:            22388

Base image class counts:
  Train: real=6400, fake=6400
  Val:   real=800, fake=4800
  Test:  real=800, fake=4797

Tensor sanity
RGB tensor shape: (3, 224, 224)
Frequency tensor shape: (1, 224, 224)
Quality feature shape: (6,)
Label: 0
Resolution bucket: 128

MODEL DEBUG
model type: <class '__main__.GatedDualBranchDeepfakeNet'>
comp

In [4]:
# !pip install kagglehub 
!pip install pandas

  Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.3 MB)


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pranabr0y/celebdf-v2image-dataset")

print("Path to dataset files:", path)

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/jovyan/.cache/kagglehub/datasets/pranabr0y/celebdf-v2image-dataset/versions/1


In [5]:
# FF++ specific code
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader

FRAMES_DIR = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames"
METADATA_CSV = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv"

df = pd.read_csv(METADATA_CSV)

print("Metadata shape:", df.shape)
print(df.head())
print(df["split"].value_counts())
print(df["label"].value_counts())
print("Missing files:", (~df["frame_path"].apply(os.path.exists)).sum())

ModuleNotFoundError: No module named 'PIL'

In [6]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.auto import tqdm

# =========================
# Config
# =========================
# DATA_ROOT = "/home/jovyan/.cache/kagglehub/datasets/pranabr0y/celebdf-v2image-dataset/versions/1/Celeb_V2"
# BATCH_SIZE = 32
# IMAGE_SIZE = 224
# EPOCHS = 10
# LR = 1e-4
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# NUM_WORKERS = 0   # use 0 first while debugging

#FF++ specific
FRAMES_DIR = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames"
METADATA_CSV = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv"
RESULTS_DIR = "./gated_research_results"

RESOLUTIONS = [128, 224, 256, 384]
IMAGE_SIZE_FOR_MODEL = 224

BATCH_SIZE = 32
EPOCHS = 8
LR = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = min(8, os.cpu_count() if os.cpu_count() is not None else 4)
SEED = 42
UNCERTAIN_THRESHOLD = 0.65
USE_AMP = True

USE_TORCH_COMPILE = False
CACHE_FEATURES = True
PIN_MEMORY = True
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

BRANCH_CHANNELS = (12, 20, 32, 48)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_gated_dual_branch_ffpp.pth")
HISTORY_JSON = os.path.join(RESULTS_DIR, "history_ffpp.json")
SUMMARY_JSON = os.path.join(RESULTS_DIR, "summary_ffpp.json")
OBSERVATIONS_TXT = os.path.join(RESULTS_DIR, "observations_ffpp.txt")
CONFUSION_MATRIX_PNG = os.path.join(RESULTS_DIR, "confusion_matrix_test_gated_ffpp.png")
CURVES_PNG = os.path.join(RESULTS_DIR, "training_curves_ffpp.png")

print("Device:", DEVICE)

# =========================
# Transforms
# =========================
train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# =========================
# Datasets
# =========================
# train_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "Train"), transform=train_tfms)
# val_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "Val"), transform=val_tfms)
# test_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "Test"), transform=val_tfms)
class FFPPFrameDataset(Dataset):
    def __init__(self, metadata_csv, split, transform=None):
        self.df = pd.read_csv(metadata_csv)
        self.df = self.df[self.df["split"].str.lower() == split.lower()].reset_index(drop=True)
        self.df = self.df[self.df["frame_path"].apply(os.path.exists)].reset_index(drop=True)
        self.transform = transform

        print(f"{split} samples: {len(self.df)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["frame_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform is not None:
            img = self.transform(img)

        return img, label

train_ds = FFPPFrameDataset(METADATA_CSV, "Train", transform=train_tfms)
val_ds = FFPPFrameDataset(METADATA_CSV, "Val", transform=val_tfms)
test_ds = FFPPFrameDataset(METADATA_CSV, "Test", transform=val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Classes:", train_ds.classes)
print("Class to idx:", train_ds.class_to_idx)
print("Train size:", len(train_ds))
print("Val size:", len(val_ds))
print("Test size:", len(test_ds))

# =========================
# Model
# =========================
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# =========================
# Train / Eval functions
# =========================
def train_one_epoch(model, loader, criterion, optimizer, device, epoch, epochs):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

        current_loss = running_loss / len(all_labels)
        current_acc = accuracy_score(all_labels, all_preds)

        pbar.set_postfix({
            "loss": f"{current_loss:.4f}",
            "acc": f"{current_acc:.4f}"
        })

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device, split_name="Val"):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    pbar = tqdm(loader, desc=f"[{split_name}]", leave=False)

    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

            current_loss = running_loss / len(all_labels)
            current_acc = accuracy_score(all_labels, all_preds)

            pbar.set_postfix({
                "loss": f"{current_loss:.4f}",
                "acc": f"{current_acc:.4f}"
            })

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    try:
        epoch_auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        epoch_auc = 0.0

    return epoch_loss, epoch_acc, epoch_auc

# =========================
# Training loop
# =========================
best_val_auc = 0.0
best_model_path = "best_resnet18_celebdf.pth"

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE, epoch, EPOCHS
    )
    val_loss, val_acc, val_auc = evaluate(
        model, val_loader, criterion, DEVICE, split_name="Val"
    )

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved best model with Val AUC: {best_val_auc:.4f}")

# =========================
# Final test
# =========================
print("\nLoading best model for final test...")
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_loss, test_acc, test_auc = evaluate(
    model, test_loader, criterion, DEVICE, split_name="Test"
)

print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test AUC: {test_auc:.4f}")

ModuleNotFoundError: No module named 'torch'

In [6]:
# Baseline results with computational efficiency 
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.auto import tqdm

# =========================
# Config
# =========================
# DATA_ROOT = "/home/jovyan/.cache/kagglehub/datasets/pranabr0y/celebdf-v2image-dataset/versions/1/Celeb_V2"
# BATCH_SIZE = 32
# IMAGE_SIZE = 224
# EPOCHS = 10
# LR = 1e-4
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# NUM_WORKERS = 0   # use 0 first while debugging

FRAMES_DIR = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames"
METADATA_CSV = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv"
RESULTS_DIR = "./gated_research_results"

RESOLUTIONS = [128, 224, 256, 384]
IMAGE_SIZE_FOR_MODEL = 224

BATCH_SIZE = 32
EPOCHS = 8
LR = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = min(8, os.cpu_count() if os.cpu_count() is not None else 4)
SEED = 42
UNCERTAIN_THRESHOLD = 0.65
USE_AMP = True

USE_TORCH_COMPILE = False
CACHE_FEATURES = True
PIN_MEMORY = True
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

BRANCH_CHANNELS = (12, 20, 32, 48)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_gated_dual_branch_ffpp.pth")
HISTORY_JSON = os.path.join(RESULTS_DIR, "history_ffpp.json")
SUMMARY_JSON = os.path.join(RESULTS_DIR, "summary_ffpp.json")
OBSERVATIONS_TXT = os.path.join(RESULTS_DIR, "observations_ffpp.txt")
CONFUSION_MATRIX_PNG = os.path.join(RESULTS_DIR, "confusion_matrix_test_gated_ffpp.png")
CURVES_PNG = os.path.join(RESULTS_DIR, "training_curves_ffpp.png")

print("Device:", DEVICE)

# =========================
# Transforms
# =========================
train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# =========================
# Datasets
# =========================
# train_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "Train"), transform=train_tfms)
# val_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "Val"), transform=val_tfms)
# test_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "Test"), transform=val_tfms)

class FFPPFrameDataset(Dataset):
    def __init__(self, metadata_csv, split, transform=None):
        self.df = pd.read_csv(metadata_csv)
        self.df = self.df[self.df["split"].str.lower() == split.lower()].reset_index(drop=True)
        self.df = self.df[self.df["frame_path"].apply(os.path.exists)].reset_index(drop=True)
        self.transform = transform

        print(f"{split} samples: {len(self.df)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["frame_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform is not None:
            img = self.transform(img)

        return img, label

train_ds = FFPPFrameDataset(METADATA_CSV, "Train", transform=train_tfms)
val_ds = FFPPFrameDataset(METADATA_CSV, "Val", transform=val_tfms)
test_ds = FFPPFrameDataset(METADATA_CSV, "Test", transform=val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# print("Classes:", train_ds.classes)
# print("Class to idx:", train_ds.class_to_idx)

print("Classes:", ["Real", "Fake"])
print("Class to idx:", {"Real": 0, "Fake": 1})
print("Train size:", len(train_ds))
print("Val size:", len(val_ds))
print("Test size:", len(test_ds))

# =========================
# Model
# =========================
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# =========================
# Train / Eval functions
# =========================
def train_one_epoch(model, loader, criterion, optimizer, device, epoch, epochs):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

        current_loss = running_loss / len(all_labels)
        current_acc = accuracy_score(all_labels, all_preds)

        pbar.set_postfix({
            "loss": f"{current_loss:.4f}",
            "acc": f"{current_acc:.4f}"
        })

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device, split_name="Val"):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    pbar = tqdm(loader, desc=f"[{split_name}]", leave=False)

    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

            current_loss = running_loss / len(all_labels)
            current_acc = accuracy_score(all_labels, all_preds)

            pbar.set_postfix({
                "loss": f"{current_loss:.4f}",
                "acc": f"{current_acc:.4f}"
            })

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    try:
        epoch_auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        epoch_auc = 0.0

    return epoch_loss, epoch_acc, epoch_auc

# =========================
# Training loop
# =========================
best_val_auc = 0.0
best_model_path = "best_resnet18_celebdf.pth"

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE, epoch, EPOCHS
    )
    val_loss, val_acc, val_auc = evaluate(
        model, val_loader, criterion, DEVICE, split_name="Val"
    )

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved best model with Val AUC: {best_val_auc:.4f}")

# =========================
# Final test
# =========================
print("\nLoading best model for final test...")
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_loss, test_acc, test_auc = evaluate(
    model, test_loader, criterion, DEVICE, split_name="Test"
)

print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test AUC: {test_auc:.4f}")

Device: cuda
Classes: ['fake', 'real']
Class to idx: {'fake': 0, 'real': 1}
Train size: 80824
Val size: 10104
Test size: 10103

===== Epoch 1/10 =====


Train Loss: 0.1152 | Train Acc: 0.9542
Val   Loss: 0.0937 | Val   Acc: 0.9641 | Val AUC: 0.9984
Saved best model with Val AUC: 0.9984

===== Epoch 2/10 =====


Train Loss: 0.0486 | Train Acc: 0.9821
Val   Loss: 0.0538 | Val   Acc: 0.9796 | Val AUC: 0.9991
Saved best model with Val AUC: 0.9991

===== Epoch 3/10 =====


Train Loss: 0.0324 | Train Acc: 0.9880
Val   Loss: 0.0304 | Val   Acc: 0.9891 | Val AUC: 0.9994
Saved best model with Val AUC: 0.9994

===== Epoch 4/10 =====


Train Loss: 0.0276 | Train Acc: 0.9899
Val   Loss: 0.0319 | Val   Acc: 0.9879 | Val AUC: 0.9993

===== Epoch 5/10 =====


...

Train Loss: 0.0167 | Train Acc: 0.9938
Val   Loss: 0.0244 | Val   Acc: 0.9918 | Val AUC: 0.9997

===== Epoch 8/10 =====


Train Loss: 0.0157 | Train Acc: 0.9943
Val   Loss: 0.0634 | Val   Acc: 0.9796 | Val AUC: 0.9996

===== Epoch 9/10 =====


Train Loss: 0.0140 | Train Acc: 0.9951
Val   Loss: 0.0181 | Val   Acc: 0.9932 | Val AUC: 0.9998
Saved best model with Val AUC: 0.9998

===== Epoch 10/10 =====


Train Loss: 0.0118 | Train Acc: 0.9957
Val   Loss: 0.0153 | Val   Acc: 0.9957 | Val AUC: 0.9999
Saved best model with Val AUC: 0.9999

Loading best model for final test...


Test Loss: 0.0136 | Test Acc: 0.9951 | Test AUC: 0.9999


In [7]:
!pip install -q thop tqdm scikit-learn matplotlib pillow opencv-python

^C
ERROR: Operation cancelled by user


In [ ]:
# running the gated version of the algorithm with creating dataset with different configuration

!pip install -q tqdm scikit-learn matplotlib pillow opencv-python thop

# =========================
# Imports
# =========================
import os
import gc
import cv2
import json
import time
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_recall_fscore_support,
    confusion_matrix
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    from thop import profile
    THOP_AVAILABLE = True
except Exception:
    THOP_AVAILABLE = False

# =========================
# Config
# =========================
# DATA_ROOT = "/home/jovyan/.cache/kagglehub/datasets/pranabr0y/celebdf-v2image-dataset/versions/1/Celeb_V2"
FRAMES_DIR = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames"
METADATA_CSV = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv"
GENERATED_ROOT = "./generated_multires_celebdf"
RESULTS_DIR = "./gated_research_results"

RESOLUTIONS = [128, 224, 256, 384]
IMAGE_SIZE_FOR_MODEL = 224

BATCH_SIZE = 32
EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
SEED = 42
UNCERTAIN_THRESHOLD = 0.65
USE_AMP = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_gated_dual_branch.pth")
HISTORY_JSON = os.path.join(RESULTS_DIR, "history.json")
SUMMARY_JSON = os.path.join(RESULTS_DIR, "summary.json")
OBSERVATIONS_TXT = os.path.join(RESULTS_DIR, "observations.txt")
CONFUSION_MATRIX_PNG = os.path.join(RESULTS_DIR, "confusion_matrix_test_gated.png")
CURVES_PNG = os.path.join(RESULTS_DIR, "training_curves.png")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)
print("THOP available:", THOP_AVAILABLE)

# =========================
# Reproducibility
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# =========================
# Helper functions
# =========================
def format_seconds(sec):
    if sec < 60:
        return f"{sec:.2f} sec"
    return f"{sec/60:.2f} min"

def list_image_files(folder):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    files = []
    for root, _, fnames in os.walk(folder):
        for fname in fnames:
            if Path(fname).suffix.lower() in exts:
                files.append(os.path.join(root, fname))
    return sorted(files)

# def find_class_dirs(split_dir):
#     real_dir, fake_dir = None, None
#     for item in os.listdir(split_dir):
#         full = os.path.join(split_dir, item)
#         if os.path.isdir(full):
#             low = item.strip().lower()
#             if low == "real":
#                 real_dir = full
#             elif low == "fake":
#                 fake_dir = full
#     return real_dir, fake_dir

# def scan_original_dataset_structure(data_root):
#     print("\nScanning original dataset structure...")
#     for split in ["Train", "Val", "Test"]:
#         split_dir = os.path.join(data_root, split)
#         if not os.path.isdir(split_dir):
#             raise FileNotFoundError(f"Missing split folder: {split_dir}")
#         real_dir, fake_dir = find_class_dirs(split_dir)
#         if real_dir is None or fake_dir is None:
#             raise FileNotFoundError(f"Could not find real/fake folders inside {split_dir}")
#         real_count = len(list_image_files(real_dir))
#         fake_count = len(list_image_files(fake_dir))
#         print(f"{split}: real={real_count}, fake={fake_count}")

# def preprocess_multires_dataset(data_root, generated_root, resolutions):
#     os.makedirs(generated_root, exist_ok=True)
#     print("\nStarting multiresolution preprocessing...")

#     for res in resolutions:
#         print(f"\nPreparing resized dataset for {res}x{res}")
#         res_root = os.path.join(generated_root, f"res_{res}")
#         os.makedirs(res_root, exist_ok=True)

#         for split in ["Train", "Val", "Test"]:
#             src_split = os.path.join(data_root, split)
#             dst_split = os.path.join(res_root, split)
#             os.makedirs(dst_split, exist_ok=True)

#             real_src, fake_src = find_class_dirs(src_split)
#             if real_src is None or fake_src is None:
#                 raise RuntimeError(f"real/fake folders missing in {src_split}")

#             for class_name, class_src in [("real", real_src), ("fake", fake_src)]:
#                 class_dst = os.path.join(dst_split, class_name)
#                 os.makedirs(class_dst, exist_ok=True)

#                 files = list_image_files(class_src)
#                 pbar = tqdm(files, desc=f"res={res} {split}/{class_name}", leave=False)
#                 for src_path in pbar:
#                     rel_name = os.path.basename(src_path)
#                     dst_path = os.path.join(class_dst, rel_name)

#                     if os.path.exists(dst_path):
#                         continue

#                     try:
#                         img = Image.open(src_path).convert("RGB")
#                         img_resized = img.resize((res, res), Image.BILINEAR)
#                         img_resized.save(dst_path, quality=95)
#                     except Exception:
#                         continue

#     print("Preprocessing complete.")

# def build_samples_from_generated(generated_root, resolutions, split):
#     samples = []
#     for res in resolutions:
#         res_root = os.path.join(generated_root, f"res_{res}", split)
#         real_dir = os.path.join(res_root, "real")
#         fake_dir = os.path.join(res_root, "fake")

#         for path in list_image_files(real_dir):
#             samples.append({
#                 "path": path,
#                 "label": 0,
#                 "resolution": res,
#                 "split": split
#             })

#         for path in list_image_files(fake_dir):
#             samples.append({
#                 "path": path,
#                 "label": 1,
#                 "resolution": res,
#                 "split": split
#             })

#     return samples

def load_ffpp_metadata(metadata_csv):
    df = pd.read_csv(metadata_csv)

    required_cols = {"frame_path", "split", "label", "label_name", "video_id"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"metadata.csv missing columns: {missing}")

    df = df[df["frame_path"].apply(os.path.exists)].reset_index(drop=True)

    print("\nLoaded FF++ metadata")
    print(df.head())
    print(f"Total usable frames: {len(df)}")

    return df


def scan_ffpp_dataset(metadata_csv):
    df = load_ffpp_metadata(metadata_csv)

    print("\nScanning FF++ dataset structure...")
    for split in ["Train", "Val", "Test"]:
        split_df = df[df["split"].str.lower() == split.lower()]
        real_count = int((split_df["label"] == 0).sum())
        fake_count = int((split_df["label"] == 1).sum())
        print(f"{split}: real={real_count}, fake={fake_count}, total={len(split_df)}")

    return df


def build_base_samples_from_metadata(df, split):
    split_df = df[df["split"].str.lower() == split.lower()].reset_index(drop=True)

    samples = []
    for _, row in split_df.iterrows():
        samples.append({
            "path": row["frame_path"],
            "label": int(row["label"]),
            "split": row["split"],
            "video_id": row["video_id"]
        })

    return samples

def print_dataset_sanity(train_samples, val_samples, test_samples):
    print("\nDataset sanity check")
    print(f"Train samples: {len(train_samples)}")
    print(f"Val samples:   {len(val_samples)}")
    print(f"Test samples:  {len(test_samples)}")

    train_counts = Counter((s["resolution"], s["label"]) for s in train_samples)
    val_counts = Counter((s["resolution"], s["label"]) for s in val_samples)
    test_counts = Counter((s["resolution"], s["label"]) for s in test_samples)

    print("\nTrain counts by resolution and class:")
    for res in RESOLUTIONS:
        print(f"  res {res}: real={train_counts[(res,0)]}, fake={train_counts[(res,1)]}")

    print("\nVal counts by resolution and class:")
    for res in RESOLUTIONS:
        print(f"  res {res}: real={val_counts[(res,0)]}, fake={val_counts[(res,1)]}")

    print("\nTest counts by resolution and class:")
    for res in RESOLUTIONS:
        print(f"  res {res}: real={test_counts[(res,0)]}, fake={test_counts[(res,1)]}")

# =========================
# Quality features and DCT
# =========================
def compute_blur_and_sharpness_from_rgb(np_rgb):
    gray = cv2.cvtColor(np_rgb, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    sharpness = float(np.var(lap))
    blur_proxy = 1.0 / (sharpness + 1e-6)
    return blur_proxy, sharpness

def compute_jpeg_compression_proxy(path):
    file_size_kb = os.path.getsize(path) / 1024.0
    ext = Path(path).suffix.lower()
    if ext in [".jpg", ".jpeg"]:
        proxy = 1.0 / (file_size_kb + 1e-6)
    else:
        proxy = 0.0
    return float(proxy)

def compute_dct_tensor(pil_img, out_size=224):
    gray = np.array(pil_img.convert("L")).astype(np.float32) / 255.0
    dct = cv2.dct(gray)
    dct = np.log1p(np.abs(dct))
    dct = (dct - dct.min()) / (dct.max() - dct.min() + 1e-6)
    dct = cv2.resize(dct, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    return torch.tensor(dct, dtype=torch.float32).unsqueeze(0)

def build_quality_features(pil_img, path, resized_resolution):
    np_rgb = np.array(pil_img.convert("RGB"))
    blur_proxy, sharpness = compute_blur_and_sharpness_from_rgb(np_rgb)
    compression_proxy = compute_jpeg_compression_proxy(path)

    res_norm = resized_resolution / 224.0
    blur_norm = min(blur_proxy * 100.0, 10.0)
    sharp_norm = min(sharpness / 500.0, 10.0)
    compression_norm = min(compression_proxy * 50.0, 10.0)
    brightness = np.mean(np_rgb) / 255.0
    contrast = np.std(np_rgb) / 255.0

    feat = np.array(
        [res_norm, blur_norm, sharp_norm, compression_norm, brightness, contrast],
        dtype=np.float32
    )
    return feat

def make_gate_targets_from_quality_features(qf_batch):
    targets = []
    qf_np = qf_batch.detach().cpu().numpy()

    for qf in qf_np:
        res_norm, blur_norm, sharp_norm, compression_norm, brightness, contrast = qf
        approx_res = res_norm * 224.0

        if approx_res >= 160 and sharp_norm >= 0.18 and compression_norm <= 0.8:
            targets.append(0)
        elif approx_res >= 128 and sharp_norm >= 0.06:
            targets.append(1)
        else:
            targets.append(2)

    return torch.tensor(targets, dtype=torch.long)

# =========================
# Dataset
# =========================
class MultiResCelebDFDataset(Dataset):
    def __init__(self, samples, image_size=224):
        self.samples = samples
        self.image_size = image_size

    def __len__(self):
        return len(self.samples)

    def pil_to_rgb_tensor(self, pil_img):
        arr = np.array(pil_img).astype(np.float32) / 255.0
        arr = np.transpose(arr, (2, 0, 1))
        tensor = torch.tensor(arr, dtype=torch.float32)

        mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)
        tensor = (tensor - mean) / std
        return tensor

    def __getitem__(self, idx):
        sample = self.samples[idx]
        path = sample["path"]
        label = sample["label"]
        resolution = sample["resolution"]

        img = Image.open(path).convert("RGB")
        quality_feat = build_quality_features(img, path, resolution)
        img_for_model = img.resize((self.image_size, self.image_size), Image.BILINEAR)

        rgb = self.pil_to_rgb_tensor(img_for_model)
        freq = compute_dct_tensor(img_for_model, out_size=self.image_size)

        return {
            "rgb": rgb,
            "freq": freq,
            "quality_features": torch.tensor(quality_feat, dtype=torch.float32),
            "label": torch.tensor(label, dtype=torch.long),
            "resolution": torch.tensor(resolution, dtype=torch.long),
            "path": path
        }

# =========================
# Model blocks
# =========================
class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=None, groups=1, act=True):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True) if act else nn.Identity()
        )

    def forward(self, x):
        return self.block(x)

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = ConvBNAct(in_ch, in_ch, kernel_size=3, stride=stride, groups=in_ch)
        self.pw = ConvBNAct(in_ch, out_ch, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        x = self.dw(x)
        x = self.pw(x)
        return x

class ExitHead(nn.Module):
    def __init__(self, in_ch, hidden_dim=64, num_classes=2, dropout=0.2):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_ch, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        feat = self.pool(x).flatten(1)
        logits = self.fc(feat)
        return logits, feat

class AdaptiveGate(nn.Module):
    def __init__(self, in_features=6, hidden_dim=32, num_routes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, num_routes)
        )

    def forward(self, qf):
        logits = self.net(qf)
        probs = torch.softmax(logits, dim=1)
        route_idx = probs.argmax(dim=1)
        return logits, probs, route_idx

class EarlyExitBranch(nn.Module):
    def __init__(self, in_ch, channels=(16, 24, 40, 64), num_classes=2, exit_hidden=64):
        super().__init__()
        c1, c2, c3, c4 = channels

        self.stem = ConvBNAct(in_ch, c1, kernel_size=3, stride=2)
        self.stage1 = nn.Sequential(
            DepthwiseSeparableBlock(c1, c2, stride=2),
            DepthwiseSeparableBlock(c2, c2, stride=1)
        )
        self.stage2 = nn.Sequential(
            DepthwiseSeparableBlock(c2, c3, stride=2),
            DepthwiseSeparableBlock(c3, c3, stride=1)
        )
        self.stage3 = nn.Sequential(
            DepthwiseSeparableBlock(c3, c4, stride=2),
            DepthwiseSeparableBlock(c4, c4, stride=1)
        )

        self.exit1 = ExitHead(c2, hidden_dim=exit_hidden, num_classes=num_classes)
        self.exit2 = ExitHead(c3, hidden_dim=exit_hidden, num_classes=num_classes)
        self.exit3 = ExitHead(c4, hidden_dim=exit_hidden, num_classes=num_classes)

    def forward(self, x):
        x = self.stem(x)

        x1 = self.stage1(x)
        e1_logits, e1_feat = self.exit1(x1)

        x2 = self.stage2(x1)
        e2_logits, e2_feat = self.exit2(x2)

        x3 = self.stage3(x2)
        e3_logits, e3_feat = self.exit3(x3)

        return {
            "exit1_logits": e1_logits,
            "exit2_logits": e2_logits,
            "exit3_logits": e3_logits,
            "deep_feat": e3_feat
        }

class GatedDualBranchDeepfakeNet(nn.Module):
    def __init__(self, branch_channels=(16, 24, 40, 64), gate_features=6, num_classes=2):
        super().__init__()
        self.spatial = EarlyExitBranch(3, channels=branch_channels, num_classes=num_classes)
        self.frequency = EarlyExitBranch(1, channels=branch_channels, num_classes=num_classes)
        self.gate = AdaptiveGate(in_features=gate_features, hidden_dim=32, num_routes=3)

        deep_dim = branch_channels[-1]
        self.fusion = nn.Sequential(
            nn.Linear(deep_dim * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, rgb, freq, qf):
        gate_logits, gate_probs, gate_routes = self.gate(qf)

        spatial_out = self.spatial(rgb)
        freq_out = self.frequency(freq)

        fused_feat = torch.cat([spatial_out["deep_feat"], freq_out["deep_feat"]], dim=1)
        fused_logits = self.fusion(fused_feat)

        return {
            "gate_logits": gate_logits,
            "gate_probs": gate_probs,
            "route_idx": gate_routes,
            "spatial_exit1": spatial_out["exit1_logits"],
            "spatial_exit2": spatial_out["exit2_logits"],
            "spatial_exit3": spatial_out["exit3_logits"],
            "freq_exit1": freq_out["exit1_logits"],
            "freq_exit2": freq_out["exit2_logits"],
            "freq_exit3": freq_out["exit3_logits"],
            "fused_logits": fused_logits
        }

    def predict(self, rgb, freq, qf, threshold=0.65, forced_route=None):
        out = self.forward(rgb, freq, qf)

        B = rgb.size(0)
        final_logits = []

        if forced_route is None:
            routes = out["route_idx"]
        else:
            routes = torch.full((B,), forced_route, dtype=torch.long, device=rgb.device)

        for i in range(B):
            route = routes[i].item()
            if route == 0:
                logits_i = 0.5 * out["spatial_exit1"][i:i+1] + 0.5 * out["freq_exit1"][i:i+1]
            elif route == 1:
                logits_i = 0.5 * out["spatial_exit2"][i:i+1] + 0.5 * out["freq_exit2"][i:i+1]
            else:
                logits_i = out["fused_logits"][i:i+1]
            final_logits.append(logits_i)

        final_logits = torch.cat(final_logits, dim=0)
        probs = torch.softmax(final_logits, dim=1)
        conf, pred = probs.max(dim=1)

        decision = []
        for c, p in zip(conf.tolist(), pred.tolist()):
            if c < threshold:
                decision.append("uncertain")
            elif p == 0:
                decision.append("real")
            else:
                decision.append("deepfake")

        return {
            "logits": final_logits,
            "probs": probs,
            "pred": pred,
            "conf": conf,
            "decision": decision,
            "routes": routes,
            "raw_outputs": out
        }

# =========================
# Loss and metrics
# =========================
def compute_total_loss(outputs, labels, gate_targets, gate_weight=0.20):
    loss = 0.0
    loss += 0.08 * F.cross_entropy(outputs["spatial_exit1"], labels)
    loss += 0.12 * F.cross_entropy(outputs["spatial_exit2"], labels)
    loss += 0.15 * F.cross_entropy(outputs["spatial_exit3"], labels)

    loss += 0.08 * F.cross_entropy(outputs["freq_exit1"], labels)
    loss += 0.12 * F.cross_entropy(outputs["freq_exit2"], labels)
    loss += 0.15 * F.cross_entropy(outputs["freq_exit3"], labels)

    loss += 0.30 * F.cross_entropy(outputs["fused_logits"], labels)
    loss += gate_weight * F.cross_entropy(outputs["gate_logits"], gate_targets)
    return loss

def compute_classification_metrics(y_true, y_pred, y_score):
    acc = accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_score)
    except Exception:
        auc = 0.0

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)

    return {
        "accuracy": float(acc),
        "auc": float(auc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "confusion_matrix": cm.tolist()
    }

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

class FastRouteWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, rgb, freq, qf):
        pred = self.model.predict(rgb, freq, qf, threshold=0.0, forced_route=0)
        return pred["logits"]

class MediumRouteWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, rgb, freq, qf):
        pred = self.model.predict(rgb, freq, qf, threshold=0.0, forced_route=1)
        return pred["logits"]

class FullRouteWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, rgb, freq, qf):
        pred = self.model.predict(rgb, freq, qf, threshold=0.0, forced_route=2)
        return pred["logits"]

def measure_flops(model, image_size=224, device="cpu"):
    if not THOP_AVAILABLE:
        return None

    rgb = torch.randn(1, 3, image_size, image_size).to(device)
    freq = torch.randn(1, 1, image_size, image_size).to(device)
    qf = torch.randn(1, 6).to(device)

    flops_info = {}
    wrappers = {
        "fast_route": FastRouteWrapper(model).to(device).eval(),
        "medium_route": MediumRouteWrapper(model).to(device).eval(),
        "full_route": FullRouteWrapper(model).to(device).eval(),
    }

    for name, wrapper in wrappers.items():
        macs, params = profile(wrapper, inputs=(rgb, freq, qf), verbose=False)
        flops_info[name] = {
            "macs": float(macs),
            "flops": float(2 * macs),
            "params": float(params)
        }

    return flops_info

def measure_inference_time(model, loader, device, forced_route=None, threshold=0.65, num_batches=20):
    model.eval()
    timings = []
    seen = 0

    with torch.no_grad():
        for batch in loader:
            rgb = batch["rgb"].to(device, non_blocking=True)
            freq = batch["freq"].to(device, non_blocking=True)
            qf = batch["quality_features"].to(device, non_blocking=True)

            if device == "cuda":
                torch.cuda.synchronize()
            start = time.time()

            _ = model.predict(rgb, freq, qf, threshold=threshold, forced_route=forced_route)

            if device == "cuda":
                torch.cuda.synchronize()
            end = time.time()

            timings.append(end - start)
            seen += 1
            if seen >= num_batches:
                break

    avg_batch = np.mean(timings)
    avg_img = avg_batch / loader.batch_size
    fps = 1.0 / avg_img if avg_img > 0 else 0.0

    return {
        "avg_batch_time_sec": float(avg_batch),
        "avg_image_time_sec": float(avg_img),
        "avg_image_time_ms": float(avg_img * 1000.0),
        "fps": float(fps)
    }

# =========================
# Train and eval
# =========================
def train_one_epoch(model, loader, optimizer, scaler, device, epoch_idx):
    model.train()

    total_loss = 0.0
    total_count = 0
    running_y_true = []
    running_y_pred = []
    running_y_score = []

    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx+1}", leave=False)

    for batch in pbar:
        rgb = batch["rgb"].to(device, non_blocking=True)
        freq = batch["freq"].to(device, non_blocking=True)
        qf = batch["quality_features"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        gate_targets = make_gate_targets_from_quality_features(qf).to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(USE_AMP and device == "cuda")):
            outputs = model(rgb, freq, qf)
            loss = compute_total_loss(outputs, labels, gate_targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            probs = torch.softmax(outputs["fused_logits"], dim=1)[:, 1]
            preds = outputs["fused_logits"].argmax(dim=1)

        bs = rgb.size(0)
        total_loss += loss.item() * bs
        total_count += bs

        running_y_true.extend(labels.detach().cpu().numpy().tolist())
        running_y_pred.extend(preds.detach().cpu().numpy().tolist())
        running_y_score.extend(probs.detach().cpu().numpy().tolist())

        current_acc = accuracy_score(running_y_true, running_y_pred)
        current_loss = total_loss / max(total_count, 1)

        pbar.set_postfix({
            "loss": f"{current_loss:.4f}",
            "acc": f"{current_acc:.4f}"
        })

    metrics = compute_classification_metrics(running_y_true, running_y_pred, running_y_score)
    metrics["loss"] = float(total_loss / max(total_count, 1))
    return metrics

@torch.no_grad()
def evaluate_model(model, loader, device, split_name="Val", forced_route=None, threshold=0.65):
    model.eval()

    all_y_true = []
    all_y_pred = []
    all_y_score = []
    all_decisions = []
    all_routes = []
    all_resolutions = []

    total_items = 0
    start_time = time.time()
    pbar = tqdm(loader, desc=f"Eval {split_name}", leave=False)

    for batch in pbar:
        rgb = batch["rgb"].to(device, non_blocking=True)
        freq = batch["freq"].to(device, non_blocking=True)
        qf = batch["quality_features"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        resolutions = batch["resolution"].cpu().numpy().tolist()

        pred_out = model.predict(rgb, freq, qf, threshold=threshold, forced_route=forced_route)

        probs_fake = pred_out["probs"][:, 1].detach().cpu().numpy().tolist()
        pred_labels = pred_out["pred"].detach().cpu().numpy().tolist()
        route_labels = pred_out["routes"].detach().cpu().numpy().tolist()

        all_y_true.extend(labels.detach().cpu().numpy().tolist())
        all_y_pred.extend(pred_labels)
        all_y_score.extend(probs_fake)
        all_decisions.extend(pred_out["decision"])
        all_routes.extend(route_labels)
        all_resolutions.extend(resolutions)

        total_items += rgb.size(0)

    elapsed = time.time() - start_time
    metrics = compute_classification_metrics(all_y_true, all_y_pred, all_y_score)

    route_counter = Counter(all_routes)
    route_percent = {
        "fast_exit1_pct": 100.0 * route_counter.get(0, 0) / max(len(all_routes), 1),
        "medium_exit2_pct": 100.0 * route_counter.get(1, 0) / max(len(all_routes), 1),
        "full_exit3_pct": 100.0 * route_counter.get(2, 0) / max(len(all_routes), 1),
    }

    uncertain_rate = 100.0 * sum(d == "uncertain" for d in all_decisions) / max(len(all_decisions), 1)

    metrics.update({
        "num_samples": int(total_items),
        "uncertain_rate_pct": float(uncertain_rate),
        "route_usage": route_percent,
        "elapsed_sec": float(elapsed),
        "elapsed_pretty": format_seconds(elapsed),
    })

    per_resolution = {}
    for res in sorted(set(all_resolutions)):
        idxs = [i for i, r in enumerate(all_resolutions) if r == res]
        y_true_res = [all_y_true[i] for i in idxs]
        y_pred_res = [all_y_pred[i] for i in idxs]
        y_score_res = [all_y_score[i] for i in idxs]
        decisions_res = [all_decisions[i] for i in idxs]
        routes_res = [all_routes[i] for i in idxs]

        res_metrics = compute_classification_metrics(y_true_res, y_pred_res, y_score_res)
        rc = Counter(routes_res)

        res_metrics["uncertain_rate_pct"] = float(100.0 * sum(d == "uncertain" for d in decisions_res) / max(len(decisions_res), 1))
        res_metrics["route_usage"] = {
            "fast_exit1_pct": 100.0 * rc.get(0, 0) / max(len(routes_res), 1),
            "medium_exit2_pct": 100.0 * rc.get(1, 0) / max(len(routes_res), 1),
            "full_exit3_pct": 100.0 * rc.get(2, 0) / max(len(routes_res), 1)
        }
        res_metrics["num_samples"] = int(len(idxs))
        per_resolution[str(res)] = res_metrics

    metrics["per_resolution"] = per_resolution
    return metrics

def print_eval_summary(name, metrics):
    print(f"\n{name}")
    print(f"  Accuracy:        {metrics['accuracy']:.4f}")
    print(f"  AUC:             {metrics['auc']:.4f}")
    print(f"  Precision:       {metrics['precision']:.4f}")
    print(f"  Recall:          {metrics['recall']:.4f}")
    print(f"  F1:              {metrics['f1']:.4f}")
    print(f"  Uncertain rate:  {metrics['uncertain_rate_pct']:.2f}%")
    print(f"  Route usage:")
    print(f"    Fast   Exit1:  {metrics['route_usage']['fast_exit1_pct']:.2f}%")
    print(f"    Medium Exit2:  {metrics['route_usage']['medium_exit2_pct']:.2f}%")
    print(f"    Full   Exit3:  {metrics['route_usage']['full_exit3_pct']:.2f}%")

def save_confusion_matrix(cm, path, title="Confusion Matrix"):
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["real", "fake"])
    plt.yticks(tick_marks, ["real", "fake"])
    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i][j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()

def plot_training_curves(history, path):
    epochs = list(range(1, len(history["train_loss"]) + 1))

    plt.figure(figsize=(14, 4))

    plt.subplot(1, 3, 1)
    plt.plot(epochs, history["train_loss"], marker="o", label="train_loss")
    plt.plot(epochs, history["val_loss"], marker="o", label="val_loss")
    plt.legend()
    plt.title("Loss")
    plt.xlabel("Epoch")

    plt.subplot(1, 3, 2)
    plt.plot(epochs, history["train_acc"], marker="o", label="train_acc")
    plt.plot(epochs, history["val_acc"], marker="o", label="val_acc")
    plt.legend()
    plt.title("Accuracy")
    plt.xlabel("Epoch")

    plt.subplot(1, 3, 3)
    plt.plot(epochs, history["val_auc"], marker="o", label="val_auc")
    plt.legend()
    plt.title("Validation AUC")
    plt.xlabel("Epoch")

    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()

def generate_observations(summary):
    lines = []
    test_gated = summary["test_gated"]
    test_fast = summary["test_fast"]
    test_medium = summary["test_medium"]
    test_full = summary["test_full"]

    lines.append("AUTOMATIC OBSERVATIONS")
    lines.append("=" * 60)
    lines.append(f"Gated test accuracy: {test_gated['accuracy']:.4f}")
    lines.append(f"Fast-only test accuracy: {test_fast['accuracy']:.4f}")
    lines.append(f"Medium-only test accuracy: {test_medium['accuracy']:.4f}")
    lines.append(f"Full-only test accuracy: {test_full['accuracy']:.4f}")
    lines.append("")

    lines.append("Route behavior")
    lines.append(
        f"Gate used Exit1 for {test_gated['route_usage']['fast_exit1_pct']:.2f}% "
        f"of test samples, Exit2 for {test_gated['route_usage']['medium_exit2_pct']:.2f}%, "
        f"and Exit3 for {test_gated['route_usage']['full_exit3_pct']:.2f}%."
    )

    if test_gated["accuracy"] >= test_full["accuracy"] - 0.01:
        lines.append("The gated system preserved almost the same accuracy as the full route while saving computation on part of the test set.")
    else:
        lines.append("The full route still outperformed the gated route, which suggests the gate may still be too aggressive on some samples.")

    if test_gated["uncertain_rate_pct"] > 10:
        lines.append("The uncertainty rate is relatively high. That usually means the threshold is conservative, which can improve trustworthiness but reduce decisiveness.")
    else:
        lines.append("The uncertainty rate is moderate, which suggests the chosen confidence threshold is reasonably balanced.")

    lines.append("")
    lines.append("Per resolution trends")
    per_res = test_gated["per_resolution"]
    sorted_res = sorted(int(k) for k in per_res.keys())

    for res in sorted_res:
        m = per_res[str(res)]
        lines.append(
            f"Resolution {res}: accuracy={m['accuracy']:.4f}, auc={m['auc']:.4f}, "
            f"fast={m['route_usage']['fast_exit1_pct']:.2f}%, "
            f"medium={m['route_usage']['medium_exit2_pct']:.2f}%, "
            f"full={m['route_usage']['full_exit3_pct']:.2f}%"
        )

    if len(sorted_res) >= 2:
        low_res = str(sorted_res[0])
        high_res = str(sorted_res[-1])
        low_acc = per_res[low_res]["accuracy"]
        high_acc = per_res[high_res]["accuracy"]
        if high_acc > low_acc:
            lines.append(f"Higher resolution inputs performed better than lower resolution inputs by {high_acc - low_acc:.4f} accuracy points.")
        else:
            lines.append("The model stayed robust across resolutions without a large performance gap.")

    low_route_full = per_res[str(sorted_res[0])]["route_usage"]["full_exit3_pct"]
    high_route_full = per_res[str(sorted_res[-1])]["route_usage"]["full_exit3_pct"]
    if low_route_full > high_route_full:
        lines.append("Lower resolution samples were routed deeper more often, which means the gate learned the intended computation policy.")
    else:
        lines.append("The gate did not route lower resolution samples deeper as strongly as expected, so the quality features or gate supervision may need tuning.")

    if "inference_benchmarks" in summary:
        bench = summary["inference_benchmarks"]
        lines.append("")
        lines.append("Speed and efficiency")
        lines.append(f"Fast route avg image time:   {bench['fast']['avg_image_time_ms']:.2f} ms")
        lines.append(f"Medium route avg image time: {bench['medium']['avg_image_time_ms']:.2f} ms")
        lines.append(f"Full route avg image time:   {bench['full']['avg_image_time_ms']:.2f} ms")
        lines.append(f"Gated route avg image time:  {bench['gated']['avg_image_time_ms']:.2f} ms")

        if bench["gated"]["avg_image_time_ms"] < bench["full"]["avg_image_time_ms"]:
            lines.append("The gated system reduced average inference time compared with always using the full route.")
        else:
            lines.append("The gated system did not reduce runtime compared with the full route, which may happen if most inputs still go to Exit3.")

    return "\n".join(lines)

# =========================
# Main pipeline
# =========================
print("\n" + "=" * 70)
print("FINAL RESEARCH PIPELINE: GATED DUAL BRANCH DEEPFAKE DETECTOR")
print("=" * 70)

# scan_original_dataset_structure(DATA_ROOT)
# preprocess_multires_dataset(DATA_ROOT, GENERATED_ROOT, RESOLUTIONS)

# train_samples = build_samples_from_generated(GENERATED_ROOT, RESOLUTIONS, "Train")
# val_samples = build_samples_from_generated(GENERATED_ROOT, RESOLUTIONS, "Val")
# test_samples = build_samples_from_generated(GENERATED_ROOT, RESOLUTIONS, "Test")

df_meta = scan_ffpp_dataset(METADATA_CSV)

train_samples = build_base_samples_from_metadata(df_meta, "Train")
val_samples = build_base_samples_from_metadata(df_meta, "Val")
test_samples = build_base_samples_from_metadata(df_meta, "Test")

print_dataset_sanity(train_samples, val_samples, test_samples)

train_ds = MultiResCelebDFDataset(train_samples, image_size=IMAGE_SIZE_FOR_MODEL)
val_ds = MultiResCelebDFDataset(val_samples, image_size=IMAGE_SIZE_FOR_MODEL)
test_ds = MultiResCelebDFDataset(test_samples, image_size=IMAGE_SIZE_FOR_MODEL)

print("\nSample image checks")
for i in random.sample(range(len(train_samples)), min(5, len(train_samples))):
    item = train_samples[i]
    raw = Image.open(item["path"]).convert("RGB")
    print(f"Path: {item['path']}")
    print(f"Raw resized image size on disk: {raw.size}, label={item['label']}, res_bucket={item['resolution']}")

sample = train_ds[0]
print("\nTensor sanity")
print("RGB tensor shape:        ", tuple(sample["rgb"].shape))
print("Frequency tensor shape:  ", tuple(sample["freq"].shape))
print("Quality feature shape:   ", tuple(sample["quality_features"].shape))
print("Quality features values: ", sample["quality_features"].tolist())
print("Label:                   ", sample["label"].item())
print("Resolution bucket:       ", sample["resolution"].item())

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

model = GatedDualBranchDeepfakeNet(
    branch_channels=(16, 24, 40, 64),
    gate_features=6,
    num_classes=2
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE == "cuda"))

total_params, trainable_params = count_parameters(model)
print("\nModel summary")
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

flops_info = measure_flops(model, image_size=IMAGE_SIZE_FOR_MODEL, device=DEVICE)
if flops_info is not None:
    print("\nApprox FLOPs")
    for k, v in flops_info.items():
        print(f"{k}: GFLOPs={v['flops'] / 1e9:.4f}, MACs={v['macs'] / 1e9:.4f}G")

history = {
    "train_loss": [],
    "train_acc": [],
    "train_auc": [],
    "val_loss": [],
    "val_acc": [],
    "val_auc": [],
    "epoch_time_sec": []
}

best_val_auc = -1.0
total_train_start = time.time()

print("\nStarting training...")
for epoch in range(EPOCHS):
    epoch_start = time.time()
    print(f"\n{'='*25} Epoch {epoch+1}/{EPOCHS} {'='*25}")

    train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, DEVICE, epoch)
    val_metrics = evaluate_model(model, val_loader, DEVICE, split_name="Val", forced_route=None, threshold=UNCERTAIN_THRESHOLD)
    epoch_time = time.time() - epoch_start

    history["train_loss"].append(train_metrics["loss"])
    history["train_acc"].append(train_metrics["accuracy"])
    history["train_auc"].append(train_metrics["auc"])
    history["val_loss"].append(0.0)
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_auc"].append(val_metrics["auc"])
    history["epoch_time_sec"].append(epoch_time)

    print("\nTrain")
    print(f"  Loss:      {train_metrics['loss']:.4f}")
    print(f"  Accuracy:  {train_metrics['accuracy']:.4f}")
    print(f"  AUC:       {train_metrics['auc']:.4f}")

    print("\nValidation")
    print(f"  Accuracy:  {val_metrics['accuracy']:.4f}")
    print(f"  AUC:       {val_metrics['auc']:.4f}")
    print(f"  Precision: {val_metrics['precision']:.4f}")
    print(f"  Recall:    {val_metrics['recall']:.4f}")
    print(f"  F1:        {val_metrics['f1']:.4f}")
    print(f"  Uncertain: {val_metrics['uncertain_rate_pct']:.2f}%")
    print(f"  Route use: fast={val_metrics['route_usage']['fast_exit1_pct']:.2f}% "
          f"medium={val_metrics['route_usage']['medium_exit2_pct']:.2f}% "
          f"full={val_metrics['route_usage']['full_exit3_pct']:.2f}%")
    print(f"  Epoch time: {format_seconds(epoch_time)}")

    if val_metrics["auc"] > best_val_auc:
        best_val_auc = val_metrics["auc"]
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Saved new best checkpoint.")

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

total_train_time = time.time() - total_train_start
print(f"\nTotal training time: {format_seconds(total_train_time)}")

plot_training_curves(history, CURVES_PNG)
with open(HISTORY_JSON, "w") as f:
    json.dump(history, f, indent=2)

print("\nLoading best checkpoint...")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))

print("\nRunning final evaluations...")
test_gated = evaluate_model(model, test_loader, DEVICE, split_name="Test Gated", forced_route=None, threshold=UNCERTAIN_THRESHOLD)
test_fast = evaluate_model(model, test_loader, DEVICE, split_name="Test Fast", forced_route=0, threshold=UNCERTAIN_THRESHOLD)
test_medium = evaluate_model(model, test_loader, DEVICE, split_name="Test Medium", forced_route=1, threshold=UNCERTAIN_THRESHOLD)
test_full = evaluate_model(model, test_loader, DEVICE, split_name="Test Full", forced_route=2, threshold=UNCERTAIN_THRESHOLD)

print_eval_summary("TEST GATED", test_gated)
print_eval_summary("TEST FAST ONLY", test_fast)
print_eval_summary("TEST MEDIUM ONLY", test_medium)
print_eval_summary("TEST FULL ONLY", test_full)

save_confusion_matrix(test_gated["confusion_matrix"], CONFUSION_MATRIX_PNG, title="Test Confusion Matrix - Gated")

inference_benchmarks = {
    "gated": measure_inference_time(model, test_loader, DEVICE, forced_route=None, threshold=UNCERTAIN_THRESHOLD, num_batches=20),
    "fast": measure_inference_time(model, test_loader, DEVICE, forced_route=0, threshold=UNCERTAIN_THRESHOLD, num_batches=20),
    "medium": measure_inference_time(model, test_loader, DEVICE, forced_route=1, threshold=UNCERTAIN_THRESHOLD, num_batches=20),
    "full": measure_inference_time(model, test_loader, DEVICE, forced_route=2, threshold=UNCERTAIN_THRESHOLD, num_batches=20),
}

print("\nInference benchmarks")
for name, bench in inference_benchmarks.items():
    print(f"  {name}: {bench['avg_image_time_ms']:.2f} ms/image, FPS={bench['fps']:.2f}")

checkpoint_size_mb = os.path.getsize(BEST_MODEL_PATH) / (1024 * 1024)

summary = {
    "config": {
        "data_root": DATA_ROOT,
        "generated_root": GENERATED_ROOT,
        "resolutions": RESOLUTIONS,
        "image_size_for_model": IMAGE_SIZE_FOR_MODEL,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "uncertain_threshold": UNCERTAIN_THRESHOLD,
        "device": DEVICE
    },
    "model_stats": {
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "checkpoint_size_mb": float(checkpoint_size_mb),
        "flops_info": flops_info
    },
    "timing": {
        "total_training_time_sec": float(total_train_time),
        "total_training_time_pretty": format_seconds(total_train_time)
    },
    "test_gated": test_gated,
    "test_fast": test_fast,
    "test_medium": test_medium,
    "test_full": test_full,
    "inference_benchmarks": inference_benchmarks
}

with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

observations = generate_observations(summary)
with open(OBSERVATIONS_TXT, "w") as f:
    f.write(observations)

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"Best checkpoint:          {BEST_MODEL_PATH}")
print(f"Checkpoint size:          {checkpoint_size_mb:.2f} MB")
print(f"Total params:             {total_params:,}")
print(f"Trainable params:         {trainable_params:,}")
print(f"Total training time:      {format_seconds(total_train_time)}")
print(f"Training curves saved:    {CURVES_PNG}")
print(f"Confusion matrix saved:   {CONFUSION_MATRIX_PNG}")
print(f"History JSON saved:       {HISTORY_JSON}")
print(f"Summary JSON saved:       {SUMMARY_JSON}")
print(f"Observations saved:       {OBSERVATIONS_TXT}")

print("\nKey test results")
print(f"Gated Accuracy:           {test_gated['accuracy']:.4f}")
print(f"Gated AUC:                {test_gated['auc']:.4f}")
print(f"Gated Uncertain rate:     {test_gated['uncertain_rate_pct']:.2f}%")
print(f"Fast route usage:         {test_gated['route_usage']['fast_exit1_pct']:.2f}%")
print(f"Medium route usage:       {test_gated['route_usage']['medium_exit2_pct']:.2f}%")
print(f"Full route usage:         {test_gated['route_usage']['full_exit3_pct']:.2f}%")

if flops_info is not None:
    print("\nApprox route GFLOPs")
    for name, info in flops_info.items():
        print(f"  {name}: {info['flops'] / 1e9:.4f} GFLOPs")

print("\nObservations preview")
print("-" * 70)
print(observations)
print("-" * 70)

In [ ]:
# faster gated dual-branch deepfake detector
# full clean version with:
# 1) NO torch.compile anywhere
# 2) no offline multires dataset duplication
# 3) train uses one random resolution per image per epoch
# 4) val/test still evaluate across all requested resolutions
# 5) safer THOP profiling
# 6) smaller backbone for faster training

# =========================
# Optional install if needed
# Uncomment if your notebook is missing packages
# =========================
!pip install tqdm scikit-learn matplotlib pillow opencv-python-headless thop pandas

# =========================
# Imports
# =========================
import os
import gc
import cv2
import json
import time
import random
import copy
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_recall_fscore_support,
    confusion_matrix
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    from thop import profile
    THOP_AVAILABLE = True
except Exception:
    THOP_AVAILABLE = False

# =========================
# Config
# =========================

# CelebDF version
# DATA_ROOT = "/home/jovyan/.cache/kagglehub/datasets/pranabr0y/celebdf-v2image-dataset/versions/1/Celeb_V2"
# RESULTS_DIR = "./gated_research_results"

# FF++ version
FRAMES_DIR = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames"
METADATA_CSV = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv"
RESULTS_DIR = "./gated_research_results_ffpp"

RESOLUTIONS = [128, 224, 256, 384]
IMAGE_SIZE_FOR_MODEL = 224

BATCH_SIZE = 32
EPOCHS = 8
LR = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = min(8, os.cpu_count() if os.cpu_count() is not None else 4)
SEED = 42
UNCERTAIN_THRESHOLD = 0.65
USE_AMP = True

# IMPORTANT: no compile in this script
USE_TORCH_COMPILE = True

CACHE_FEATURES = True
PIN_MEMORY = True
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

# smaller backbone for speed
BRANCH_CHANNELS = (12, 20, 32, 48)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# CelebDF outputs
# BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_gated_dual_branch.pth")
# HISTORY_JSON = os.path.join(RESULTS_DIR, "history.json")
# SUMMARY_JSON = os.path.join(RESULTS_DIR, "summary.json")
# OBSERVATIONS_TXT = os.path.join(RESULTS_DIR, "observations.txt")
# CONFUSION_MATRIX_PNG = os.path.join(RESULTS_DIR, "confusion_matrix_test_gated.png")
# CURVES_PNG = os.path.join(RESULTS_DIR, "training_curves.png")

# FF++ outputs
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_gated_dual_branch_ffpp.pth")
HISTORY_JSON = os.path.join(RESULTS_DIR, "history_ffpp.json")
SUMMARY_JSON = os.path.join(RESULTS_DIR, "summary_ffpp.json")
OBSERVATIONS_TXT = os.path.join(RESULTS_DIR, "observations_ffpp.txt")
CONFUSION_MATRIX_PNG = os.path.join(RESULTS_DIR, "confusion_matrix_test_gated_ffpp.png")
CURVES_PNG = os.path.join(RESULTS_DIR, "training_curves_ffpp.png")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)
print("THOP available:", THOP_AVAILABLE)
print("USE_TORCH_COMPILE:", USE_TORCH_COMPILE)
print("FRAMES_DIR:", FRAMES_DIR)
print("METADATA_CSV:", METADATA_CSV)

# =========================
# Reproducibility + speed
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# =========================
# Helper functions
# =========================
def format_seconds(sec):
    if sec < 60:
        return f"{sec:.2f} sec"
    return f"{sec/60:.2f} min"

def list_image_files(folder):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    files = []
    for root, _, fnames in os.walk(folder):
        for fname in fnames:
            if Path(fname).suffix.lower() in exts:
                files.append(os.path.join(root, fname))
    return sorted(files)

# -------------------------
# CelebDF folder-based helpers
# kept for reference / GitHub history
# -------------------------
# def find_class_dirs(split_dir):
#     real_dir, fake_dir = None, None
#     for item in os.listdir(split_dir):
#         full = os.path.join(split_dir, item)
#         if os.path.isdir(full):
#             low = item.strip().lower()
#             if low == "real":
#                 real_dir = full
#             elif low == "fake":
#                 fake_dir = full
#     return real_dir, fake_dir

# def scan_original_dataset_structure(data_root):
#     print("\nScanning original dataset structure...")
#     for split in ["Train", "Val", "Test"]:
#         split_dir = os.path.join(data_root, split)
#         if not os.path.isdir(split_dir):
#             raise FileNotFoundError(f"Missing split folder: {split_dir}")
#         real_dir, fake_dir = find_class_dirs(split_dir)
#         if real_dir is None or fake_dir is None:
#             raise FileNotFoundError(f"Could not find real/fake folders inside {split_dir}")
#         real_count = len(list_image_files(real_dir))
#         fake_count = len(list_image_files(fake_dir))
#         print(f"{split}: real={real_count}, fake={fake_count}")

# def build_base_samples(data_root, split):
#     split_dir = os.path.join(data_root, split)
#     real_dir, fake_dir = find_class_dirs(split_dir)

#     if real_dir is None or fake_dir is None:
#         raise RuntimeError(f"real/fake folders missing in {split_dir}")

#     samples = []

#     for path in list_image_files(real_dir):
#         samples.append({
#             "path": path,
#             "label": 0,
#             "split": split
#         })

#     for path in list_image_files(fake_dir):
#         samples.append({
#             "path": path,
#             "label": 1,
#             "split": split
#         })

#     return samples

# -------------------------
# FF++ metadata-based helpers
# -------------------------
def load_ffpp_metadata(metadata_csv):
    df = pd.read_csv(metadata_csv)

    required_cols = {"frame_path", "split", "label", "label_name", "video_id"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"metadata.csv missing columns: {missing}")

    df = df[df["frame_path"].apply(os.path.exists)].reset_index(drop=True)

    print("\nLoaded FF++ metadata")
    print(df.head())
    print(f"Total usable frames: {len(df)}")

    return df


def scan_ffpp_dataset(metadata_csv):
    df = load_ffpp_metadata(metadata_csv)

    print("\nScanning FF++ dataset structure...")
    for split in ["Train", "Val", "Test"]:
        split_df = df[df["split"].str.lower() == split.lower()]
        real_count = int((split_df["label"] == 0).sum())
        fake_count = int((split_df["label"] == 1).sum())
        print(f"{split}: real={real_count}, fake={fake_count}, total={len(split_df)}")

    return df


def build_base_samples_from_metadata(df, split):
    split_df = df[df["split"].str.lower() == split.lower()].reset_index(drop=True)

    samples = []
    for _, row in split_df.iterrows():
        samples.append({
            "path": row["frame_path"],
            "label": int(row["label"]),
            "split": row["split"],
            "video_id": row["video_id"]
        })

    return samples

def print_dataset_sanity(train_samples, val_samples, test_samples):
    print("\nDataset sanity check")
    print(f"Train base images: {len(train_samples)}")
    print(f"Val base images:   {len(val_samples)}")
    print(f"Test base images:  {len(test_samples)}")
    print(f"Train effective samples per epoch: {len(train_samples)}")
    print(f"Val effective samples:             {len(val_samples) * len(RESOLUTIONS)}")
    print(f"Test effective samples:            {len(test_samples) * len(RESOLUTIONS)}")

    train_counts = Counter(s["label"] for s in train_samples)
    val_counts = Counter(s["label"] for s in val_samples)
    test_counts = Counter(s["label"] for s in test_samples)

    print("\nBase image class counts:")
    print(f"  Train: real={train_counts[0]}, fake={train_counts[1]}")
    print(f"  Val:   real={val_counts[0]}, fake={val_counts[1]}")
    print(f"  Test:  real={test_counts[0]}, fake={test_counts[1]}")

# =========================
# Quality features and DCT
# =========================
def compute_blur_and_sharpness_from_rgb(np_rgb):
    gray = cv2.cvtColor(np_rgb, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    sharpness = float(np.var(lap))
    blur_proxy = 1.0 / (sharpness + 1e-6)
    return blur_proxy, sharpness

def compute_jpeg_compression_proxy(path):
    file_size_kb = os.path.getsize(path) / 1024.0
    ext = Path(path).suffix.lower()
    if ext in [".jpg", ".jpeg"]:
        proxy = 1.0 / (file_size_kb + 1e-6)
    else:
        proxy = 0.0
    return float(proxy)

def compute_dct_tensor(pil_img, out_size=224):
    gray = np.array(pil_img.convert("L")).astype(np.float32) / 255.0
    dct = cv2.dct(gray)
    dct = np.log1p(np.abs(dct))
    dct = (dct - dct.min()) / (dct.max() - dct.min() + 1e-6)
    dct = cv2.resize(dct, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    return torch.tensor(dct, dtype=torch.float32).unsqueeze(0)

def build_quality_features(pil_img, path, resized_resolution):
    np_rgb = np.array(pil_img.convert("RGB"))
    blur_proxy, sharpness = compute_blur_and_sharpness_from_rgb(np_rgb)
    compression_proxy = compute_jpeg_compression_proxy(path)

    res_norm = resized_resolution / 224.0
    blur_norm = min(blur_proxy * 100.0, 10.0)
    sharp_norm = min(sharpness / 500.0, 10.0)
    compression_norm = min(compression_proxy * 50.0, 10.0)
    brightness = np.mean(np_rgb) / 255.0
    contrast = np.std(np_rgb) / 255.0

    feat = np.array(
        [res_norm, blur_norm, sharp_norm, compression_norm, brightness, contrast],
        dtype=np.float32
    )
    return feat

def make_gate_targets_from_quality_features(qf_batch):
    targets = []
    qf_np = qf_batch.detach().cpu().numpy()

    for qf in qf_np:
        res_norm, blur_norm, sharp_norm, compression_norm, brightness, contrast = qf
        approx_res = res_norm * 224.0

        if approx_res >= 160 and sharp_norm >= 0.18 and compression_norm <= 0.8:
            targets.append(0)
        elif approx_res >= 128 and sharp_norm >= 0.06:
            targets.append(1)
        else:
            targets.append(2)

    return torch.tensor(targets, dtype=torch.long)

# =========================
# Dataset
# =========================
class MultiResCelebDFDataset(Dataset):
    def __init__(
        self,
        base_samples,
        image_size=224,
        resolutions=(128, 224, 256, 384),
        mode="train_random",
        cache_features=True
    ):
        """
        mode:
          - train_random: each original image appears once per epoch with random resolution
          - eval_all: each original image is expanded across all resolutions
        """
        self.base_samples = base_samples
        self.image_size = image_size
        self.resolutions = list(resolutions)
        self.mode = mode
        self.cache_features = cache_features
        self.cache = {}

        if self.mode == "eval_all":
            self.samples = []
            for s in self.base_samples:
                for res in self.resolutions:
                    self.samples.append({
                        "path": s["path"],
                        "label": s["label"],
                        "split": s["split"],
                        "resolution": res
                    })
        else:
            self.samples = self.base_samples

        self.mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)

    def __len__(self):
        return len(self.samples)

    def pil_to_rgb_tensor(self, pil_img):
        arr = np.array(pil_img).astype(np.float32) / 255.0
        arr = np.transpose(arr, (2, 0, 1))
        tensor = torch.from_numpy(arr).float()
        tensor = (tensor - self.mean) / self.std
        return tensor

    def _make_item(self, path, label, resolution):
        cache_key = (path, resolution)

        if self.cache_features and cache_key in self.cache:
            rgb, freq, quality_feat = self.cache[cache_key]
        else:
            img = Image.open(path).convert("RGB")
            img_res = img.resize((resolution, resolution), Image.BILINEAR)

            quality_feat_np = build_quality_features(img_res, path, resolution)
            img_for_model = img_res.resize((self.image_size, self.image_size), Image.BILINEAR)

            rgb = self.pil_to_rgb_tensor(img_for_model)
            freq = compute_dct_tensor(img_for_model, out_size=self.image_size)
            quality_feat = torch.tensor(quality_feat_np, dtype=torch.float32)

            if self.cache_features:
                self.cache[cache_key] = (rgb, freq, quality_feat)

        return {
            "rgb": rgb,
            "freq": freq,
            "quality_features": quality_feat,
            "label": torch.tensor(label, dtype=torch.long),
            "resolution": torch.tensor(resolution, dtype=torch.long),
            "path": path
        }

    def __getitem__(self, idx):
        sample = self.samples[idx]
        path = sample["path"]
        label = sample["label"]

        if self.mode == "train_random":
            resolution = random.choice(self.resolutions)
        else:
            resolution = sample["resolution"]

        return self._make_item(path, label, resolution)

# =========================
# Model blocks
# =========================
class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=None, groups=1, act=True):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True) if act else nn.Identity()
        )

    def forward(self, x):
        return self.block(x)

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = ConvBNAct(in_ch, in_ch, kernel_size=3, stride=stride, groups=in_ch)
        self.pw = ConvBNAct(in_ch, out_ch, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        x = self.dw(x)
        x = self.pw(x)
        return x

class ExitHead(nn.Module):
    def __init__(self, in_ch, hidden_dim=64, num_classes=2, dropout=0.2):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_ch, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        feat = self.pool(x).flatten(1)
        logits = self.fc(feat)
        return logits, feat

class AdaptiveGate(nn.Module):
    def __init__(self, in_features=6, hidden_dim=32, num_routes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, num_routes)
        )

    def forward(self, qf):
        logits = self.net(qf)
        probs = torch.softmax(logits, dim=1)
        route_idx = probs.argmax(dim=1)
        return logits, probs, route_idx

class EarlyExitBranch(nn.Module):
    def __init__(self, in_ch, channels=(12, 20, 32, 48), num_classes=2, exit_hidden=48):
        super().__init__()
        c1, c2, c3, c4 = channels

        self.stem = ConvBNAct(in_ch, c1, kernel_size=3, stride=2)
        self.stage1 = nn.Sequential(
            DepthwiseSeparableBlock(c1, c2, stride=2),
            DepthwiseSeparableBlock(c2, c2, stride=1)
        )
        self.stage2 = nn.Sequential(
            DepthwiseSeparableBlock(c2, c3, stride=2),
            DepthwiseSeparableBlock(c3, c3, stride=1)
        )
        self.stage3 = nn.Sequential(
            DepthwiseSeparableBlock(c3, c4, stride=2),
            DepthwiseSeparableBlock(c4, c4, stride=1)
        )

        self.exit1 = ExitHead(c2, hidden_dim=exit_hidden, num_classes=num_classes)
        self.exit2 = ExitHead(c3, hidden_dim=exit_hidden, num_classes=num_classes)
        self.exit3 = ExitHead(c4, hidden_dim=exit_hidden, num_classes=num_classes)

    def forward(self, x):
        x = self.stem(x)

        x1 = self.stage1(x)
        e1_logits, e1_feat = self.exit1(x1)

        x2 = self.stage2(x1)
        e2_logits, e2_feat = self.exit2(x2)

        x3 = self.stage3(x2)
        e3_logits, e3_feat = self.exit3(x3)

        return {
            "exit1_logits": e1_logits,
            "exit2_logits": e2_logits,
            "exit3_logits": e3_logits,
            "deep_feat": e3_feat
        }

class GatedDualBranchDeepfakeNet(nn.Module):
    def __init__(self, branch_channels=(12, 20, 32, 48), gate_features=6, num_classes=2):
        super().__init__()
        self.spatial = EarlyExitBranch(3, channels=branch_channels, num_classes=num_classes, exit_hidden=48)
        self.frequency = EarlyExitBranch(1, channels=branch_channels, num_classes=num_classes, exit_hidden=48)
        self.gate = AdaptiveGate(in_features=gate_features, hidden_dim=32, num_routes=3)

        deep_dim = branch_channels[-1]
        self.fusion = nn.Sequential(
            nn.Linear(deep_dim * 2, 96),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(96, num_classes)
        )

    def forward(self, rgb, freq, qf):
        gate_logits, gate_probs, gate_routes = self.gate(qf)

        spatial_out = self.spatial(rgb)
        freq_out = self.frequency(freq)

        fused_feat = torch.cat([spatial_out["deep_feat"], freq_out["deep_feat"]], dim=1)
        fused_logits = self.fusion(fused_feat)

        return {
            "gate_logits": gate_logits,
            "gate_probs": gate_probs,
            "route_idx": gate_routes,
            "spatial_exit1": spatial_out["exit1_logits"],
            "spatial_exit2": spatial_out["exit2_logits"],
            "spatial_exit3": spatial_out["exit3_logits"],
            "freq_exit1": freq_out["exit1_logits"],
            "freq_exit2": freq_out["exit2_logits"],
            "freq_exit3": freq_out["exit3_logits"],
            "fused_logits": fused_logits
        }

    def predict(self, rgb, freq, qf, threshold=0.65, forced_route=None):
        out = self.forward(rgb, freq, qf)

        B = rgb.size(0)
        final_logits = []

        if forced_route is None:
            routes = out["route_idx"]
        else:
            routes = torch.full((B,), forced_route, dtype=torch.long, device=rgb.device)

        for i in range(B):
            route = routes[i].item()
            if route == 0:
                logits_i = 0.5 * out["spatial_exit1"][i:i+1] + 0.5 * out["freq_exit1"][i:i+1]
            elif route == 1:
                logits_i = 0.5 * out["spatial_exit2"][i:i+1] + 0.5 * out["freq_exit2"][i:i+1]
            else:
                logits_i = out["fused_logits"][i:i+1]
            final_logits.append(logits_i)

        final_logits = torch.cat(final_logits, dim=0)
        probs = torch.softmax(final_logits, dim=1)
        conf, pred = probs.max(dim=1)

        decision = []
        for c, p in zip(conf.tolist(), pred.tolist()):
            if c < threshold:
                decision.append("uncertain")
            elif p == 0:
                decision.append("real")
            else:
                decision.append("deepfake")

        return {
            "logits": final_logits,
            "probs": probs,
            "pred": pred,
            "conf": conf,
            "decision": decision,
            "routes": routes,
            "raw_outputs": out
        }

# =========================
# Loss and metrics
# =========================
def compute_total_loss(outputs, labels, gate_targets, gate_weight=0.20):
    loss = 0.0
    loss += 0.08 * F.cross_entropy(outputs["spatial_exit1"], labels)
    loss += 0.12 * F.cross_entropy(outputs["spatial_exit2"], labels)
    loss += 0.15 * F.cross_entropy(outputs["spatial_exit3"], labels)

    loss += 0.08 * F.cross_entropy(outputs["freq_exit1"], labels)
    loss += 0.12 * F.cross_entropy(outputs["freq_exit2"], labels)
    loss += 0.15 * F.cross_entropy(outputs["freq_exit3"], labels)

    loss += 0.30 * F.cross_entropy(outputs["fused_logits"], labels)
    loss += gate_weight * F.cross_entropy(outputs["gate_logits"], gate_targets)
    return loss

def compute_classification_metrics(y_true, y_pred, y_score):
    acc = accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_score)
    except Exception:
        auc = 0.0

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)

    return {
        "accuracy": float(acc),
        "auc": float(auc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "confusion_matrix": cm.tolist()
    }

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

class FastRouteWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, rgb, freq, qf):
        pred = self.model.predict(rgb, freq, qf, threshold=0.0, forced_route=0)
        return pred["logits"]

class MediumRouteWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, rgb, freq, qf):
        pred = self.model.predict(rgb, freq, qf, threshold=0.0, forced_route=1)
        return pred["logits"]

class FullRouteWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, rgb, freq, qf):
        pred = self.model.predict(rgb, freq, qf, threshold=0.0, forced_route=2)
        return pred["logits"]

def remove_thop_buffers(module):
    for m in module.modules():
        if hasattr(m, "total_ops"):
            try:
                delattr(m, "total_ops")
            except Exception:
                pass
        if hasattr(m, "total_params"):
            try:
                delattr(m, "total_params")
            except Exception:
                pass

def measure_flops(model, image_size=224, device="cpu"):
    if not THOP_AVAILABLE:
        return None

    model.eval()
    rgb = torch.randn(1, 3, image_size, image_size).to(device)
    freq = torch.randn(1, 1, image_size, image_size).to(device)
    qf = torch.randn(1, 6).to(device)

    flops_info = {}

    wrappers = {
        "fast_route": FastRouteWrapper(copy.deepcopy(model)).to(device).eval(),
        "medium_route": MediumRouteWrapper(copy.deepcopy(model)).to(device).eval(),
        "full_route": FullRouteWrapper(copy.deepcopy(model)).to(device).eval(),
    }

    for name, wrapper in wrappers.items():
        remove_thop_buffers(wrapper)
        try:
            macs, params = profile(wrapper, inputs=(rgb, freq, qf), verbose=False)
            flops_info[name] = {
                "macs": float(macs),
                "flops": float(2 * macs),
                "params": float(params)
            }
        except Exception as e:
            print(f"THOP profiling failed for {name}: {e}")
            flops_info[name] = None
        finally:
            remove_thop_buffers(wrapper)

    return flops_info

def move_batch_to_device(batch, device, use_channels_last=False):
    rgb = batch["rgb"].to(device, non_blocking=True)
    freq = batch["freq"].to(device, non_blocking=True)
    qf = batch["quality_features"].to(device, non_blocking=True)
    labels = batch["label"].to(device, non_blocking=True)

    if use_channels_last:
        rgb = rgb.contiguous(memory_format=torch.channels_last)
        freq = freq.contiguous(memory_format=torch.channels_last)

    return rgb, freq, qf, labels

def measure_inference_time(model, loader, device, forced_route=None, threshold=0.65, num_batches=20):
    model.eval()
    timings = []
    seen = 0

    with torch.no_grad():
        for batch in loader:
            rgb = batch["rgb"].to(device, non_blocking=True)
            freq = batch["freq"].to(device, non_blocking=True)
            qf = batch["quality_features"].to(device, non_blocking=True)

            if device == "cuda":
                rgb = rgb.contiguous(memory_format=torch.channels_last)
                freq = freq.contiguous(memory_format=torch.channels_last)
                torch.cuda.synchronize()

            start = time.time()
            _ = model.predict(rgb, freq, qf, threshold=threshold, forced_route=forced_route)

            if device == "cuda":
                torch.cuda.synchronize()
            end = time.time()

            timings.append(end - start)
            seen += 1
            if seen >= num_batches:
                break

    avg_batch = np.mean(timings)
    avg_img = avg_batch / loader.batch_size
    fps = 1.0 / avg_img if avg_img > 0 else 0.0

    return {
        "avg_batch_time_sec": float(avg_batch),
        "avg_image_time_sec": float(avg_img),
        "avg_image_time_ms": float(avg_img * 1000.0),
        "fps": float(fps)
    }

# =========================
# Train and eval
# =========================
def train_one_epoch(model, loader, optimizer, scaler, device, epoch_idx):
    model.train()

    total_loss = 0.0
    total_count = 0
    running_y_true = []
    running_y_pred = []
    running_y_score = []

    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx+1}", leave=False)

    for batch in pbar:
        rgb, freq, qf, labels = move_batch_to_device(
            batch, device, use_channels_last=(device == "cuda")
        )

        gate_targets = make_gate_targets_from_quality_features(qf).to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(USE_AMP and device == "cuda")):
            outputs = model(rgb, freq, qf)
            loss = compute_total_loss(outputs, labels, gate_targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            probs = torch.softmax(outputs["fused_logits"], dim=1)[:, 1]
            preds = outputs["fused_logits"].argmax(dim=1)

        bs = rgb.size(0)
        total_loss += loss.item() * bs
        total_count += bs

        running_y_true.extend(labels.detach().cpu().numpy().tolist())
        running_y_pred.extend(preds.detach().cpu().numpy().tolist())
        running_y_score.extend(probs.detach().cpu().numpy().tolist())

        current_acc = accuracy_score(running_y_true, running_y_pred)
        current_loss = total_loss / max(total_count, 1)

        pbar.set_postfix({
            "loss": f"{current_loss:.4f}",
            "acc": f"{current_acc:.4f}"
        })

    metrics = compute_classification_metrics(running_y_true, running_y_pred, running_y_score)
    metrics["loss"] = float(total_loss / max(total_count, 1))
    return metrics

@torch.no_grad()
def evaluate_model(model, loader, device, split_name="Val", forced_route=None, threshold=0.65):
    model.eval()

    all_y_true = []
    all_y_pred = []
    all_y_score = []
    all_decisions = []
    all_routes = []
    all_resolutions = []

    total_items = 0
    start_time = time.time()
    pbar = tqdm(loader, desc=f"Eval {split_name}", leave=False)

    for batch in pbar:
        rgb = batch["rgb"].to(device, non_blocking=True)
        freq = batch["freq"].to(device, non_blocking=True)
        qf = batch["quality_features"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        resolutions = batch["resolution"].cpu().numpy().tolist()

        if device == "cuda":
            rgb = rgb.contiguous(memory_format=torch.channels_last)
            freq = freq.contiguous(memory_format=torch.channels_last)

        pred_out = model.predict(rgb, freq, qf, threshold=threshold, forced_route=forced_route)

        probs_fake = pred_out["probs"][:, 1].detach().cpu().numpy().tolist()
        pred_labels = pred_out["pred"].detach().cpu().numpy().tolist()
        route_labels = pred_out["routes"].detach().cpu().numpy().tolist()

        all_y_true.extend(labels.detach().cpu().numpy().tolist())
        all_y_pred.extend(pred_labels)
        all_y_score.extend(probs_fake)
        all_decisions.extend(pred_out["decision"])
        all_routes.extend(route_labels)
        all_resolutions.extend(resolutions)

        total_items += rgb.size(0)

    elapsed = time.time() - start_time
    metrics = compute_classification_metrics(all_y_true, all_y_pred, all_y_score)

    route_counter = Counter(all_routes)
    route_percent = {
        "fast_exit1_pct": 100.0 * route_counter.get(0, 0) / max(len(all_routes), 1),
        "medium_exit2_pct": 100.0 * route_counter.get(1, 0) / max(len(all_routes), 1),
        "full_exit3_pct": 100.0 * route_counter.get(2, 0) / max(len(all_routes), 1),
    }

    uncertain_rate = 100.0 * sum(d == "uncertain" for d in all_decisions) / max(len(all_decisions), 1)

    metrics.update({
        "num_samples": int(total_items),
        "uncertain_rate_pct": float(uncertain_rate),
        "route_usage": route_percent,
        "elapsed_sec": float(elapsed),
        "elapsed_pretty": format_seconds(elapsed),
    })

    per_resolution = {}
    for res in sorted(set(all_resolutions)):
        idxs = [i for i, r in enumerate(all_resolutions) if r == res]
        y_true_res = [all_y_true[i] for i in idxs]
        y_pred_res = [all_y_pred[i] for i in idxs]
        y_score_res = [all_y_score[i] for i in idxs]
        decisions_res = [all_decisions[i] for i in idxs]
        routes_res = [all_routes[i] for i in idxs]

        res_metrics = compute_classification_metrics(y_true_res, y_pred_res, y_score_res)
        rc = Counter(routes_res)

        res_metrics["uncertain_rate_pct"] = float(
            100.0 * sum(d == "uncertain" for d in decisions_res) / max(len(decisions_res), 1)
        )
        res_metrics["route_usage"] = {
            "fast_exit1_pct": 100.0 * rc.get(0, 0) / max(len(routes_res), 1),
            "medium_exit2_pct": 100.0 * rc.get(1, 0) / max(len(routes_res), 1),
            "full_exit3_pct": 100.0 * rc.get(2, 0) / max(len(routes_res), 1)
        }
        res_metrics["num_samples"] = int(len(idxs))
        per_resolution[str(res)] = res_metrics

    metrics["per_resolution"] = per_resolution
    return metrics

def print_eval_summary(name, metrics):
    print(f"\n{name}")
    print(f"  Accuracy:        {metrics['accuracy']:.4f}")
    print(f"  AUC:             {metrics['auc']:.4f}")
    print(f"  Precision:       {metrics['precision']:.4f}")
    print(f"  Recall:          {metrics['recall']:.4f}")
    print(f"  F1:              {metrics['f1']:.4f}")
    print(f"  Uncertain rate:  {metrics['uncertain_rate_pct']:.2f}%")
    print(f"  Route usage:")
    print(f"    Fast   Exit1:  {metrics['route_usage']['fast_exit1_pct']:.2f}%")
    print(f"    Medium Exit2:  {metrics['route_usage']['medium_exit2_pct']:.2f}%")
    print(f"    Full   Exit3:  {metrics['route_usage']['full_exit3_pct']:.2f}%")

def save_confusion_matrix(cm, path, title="Confusion Matrix"):
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["real", "fake"])
    plt.yticks(tick_marks, ["real", "fake"])
    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i][j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()

def plot_training_curves(history, path):
    epochs = list(range(1, len(history["train_loss"]) + 1))

    plt.figure(figsize=(14, 4))

    plt.subplot(1, 3, 1)
    plt.plot(epochs, history["train_loss"], marker="o", label="train_loss")
    plt.plot(epochs, history["val_loss"], marker="o", label="val_loss")
    plt.legend()
    plt.title("Loss")
    plt.xlabel("Epoch")

    plt.subplot(1, 3, 2)
    plt.plot(epochs, history["train_acc"], marker="o", label="train_acc")
    plt.plot(epochs, history["val_acc"], marker="o", label="val_acc")
    plt.legend()
    plt.title("Accuracy")
    plt.xlabel("Epoch")

    plt.subplot(1, 3, 3)
    plt.plot(epochs, history["val_auc"], marker="o", label="val_auc")
    plt.legend()
    plt.title("Validation AUC")
    plt.xlabel("Epoch")

    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()

def generate_observations(summary):
    lines = []
    test_gated = summary["test_gated"]
    test_fast = summary["test_fast"]
    test_medium = summary["test_medium"]
    test_full = summary["test_full"]

    lines.append("AUTOMATIC OBSERVATIONS")
    lines.append("=" * 60)
    lines.append(f"Gated test accuracy: {test_gated['accuracy']:.4f}")
    lines.append(f"Fast-only test accuracy: {test_fast['accuracy']:.4f}")
    lines.append(f"Medium-only test accuracy: {test_medium['accuracy']:.4f}")
    lines.append(f"Full-only test accuracy: {test_full['accuracy']:.4f}")
    lines.append("")

    lines.append("Route behavior")
    lines.append(
        f"Gate used Exit1 for {test_gated['route_usage']['fast_exit1_pct']:.2f}% "
        f"of test samples, Exit2 for {test_gated['route_usage']['medium_exit2_pct']:.2f}%, "
        f"and Exit3 for {test_gated['route_usage']['full_exit3_pct']:.2f}%."
    )

    if test_gated["accuracy"] >= test_full["accuracy"] - 0.01:
        lines.append("The gated system preserved almost the same accuracy as the full route while saving computation on part of the test set.")
    else:
        lines.append("The full route still outperformed the gated route, which suggests the gate may still be too aggressive on some samples.")

    if test_gated["uncertain_rate_pct"] > 10:
        lines.append("The uncertainty rate is relatively high. That usually means the threshold is conservative, which can improve trustworthiness but reduce decisiveness.")
    else:
        lines.append("The uncertainty rate is moderate, which suggests the chosen confidence threshold is reasonably balanced.")

    lines.append("")
    lines.append("Per resolution trends")
    per_res = test_gated["per_resolution"]
    sorted_res = sorted(int(k) for k in per_res.keys())

    for res in sorted_res:
        m = per_res[str(res)]
        lines.append(
            f"Resolution {res}: accuracy={m['accuracy']:.4f}, auc={m['auc']:.4f}, "
            f"fast={m['route_usage']['fast_exit1_pct']:.2f}%, "
            f"medium={m['route_usage']['medium_exit2_pct']:.2f}%, "
            f"full={m['route_usage']['full_exit3_pct']:.2f}%"
        )

    if len(sorted_res) >= 2:
        low_res = str(sorted_res[0])
        high_res = str(sorted_res[-1])
        low_acc = per_res[low_res]["accuracy"]
        high_acc = per_res[high_res]["accuracy"]
        if high_acc > low_acc:
            lines.append(f"Higher resolution inputs performed better than lower resolution inputs by {high_acc - low_acc:.4f} accuracy points.")
        else:
            lines.append("The model stayed robust across resolutions without a large performance gap.")

    low_route_full = per_res[str(sorted_res[0])]["route_usage"]["full_exit3_pct"]
    high_route_full = per_res[str(sorted_res[-1])]["route_usage"]["full_exit3_pct"]
    if low_route_full > high_route_full:
        lines.append("Lower resolution samples were routed deeper more often, which means the gate learned the intended computation policy.")
    else:
        lines.append("The gate did not route lower resolution samples deeper as strongly as expected, so the quality features or gate supervision may need tuning.")

    if "inference_benchmarks" in summary:
        bench = summary["inference_benchmarks"]
        lines.append("")
        lines.append("Speed and efficiency")
        lines.append(f"Fast route avg image time:   {bench['fast']['avg_image_time_ms']:.2f} ms")
        lines.append(f"Medium route avg image time: {bench['medium']['avg_image_time_ms']:.2f} ms")
        lines.append(f"Full route avg image time:   {bench['full']['avg_image_time_ms']:.2f} ms")
        lines.append(f"Gated route avg image time:  {bench['gated']['avg_image_time_ms']:.2f} ms")

        if bench["gated"]["avg_image_time_ms"] < bench["full"]["avg_image_time_ms"]:
            lines.append("The gated system reduced average inference time compared with always using the full route.")
        else:
            lines.append("The gated system did not reduce runtime compared with the full route, which may happen if most inputs still go to Exit3.")

    return "\n".join(lines)

# =========================
# Main pipeline
# =========================
print("\n" + "=" * 70)
print("FINAL RESEARCH PIPELINE: FASTER GATED DUAL BRANCH DEEPFAKE DETECTOR")
print("=" * 70)

# CelebDF version
# scan_original_dataset_structure(DATA_ROOT)
# train_samples = build_base_samples(DATA_ROOT, "Train")
# val_samples = build_base_samples(DATA_ROOT, "Val")
# test_samples = build_base_samples(DATA_ROOT, "Test")

# FF++ version
df_meta = scan_ffpp_dataset(METADATA_CSV)
train_samples = build_base_samples_from_metadata(df_meta, "Train")
val_samples = build_base_samples_from_metadata(df_meta, "Val")
test_samples = build_base_samples_from_metadata(df_meta, "Test")

print_dataset_sanity(train_samples, val_samples, test_samples)

train_ds = MultiResCelebDFDataset(
    train_samples,
    image_size=IMAGE_SIZE_FOR_MODEL,
    resolutions=RESOLUTIONS,
    mode="train_random",
    cache_features=CACHE_FEATURES
)
val_ds = MultiResCelebDFDataset(
    val_samples,
    image_size=IMAGE_SIZE_FOR_MODEL,
    resolutions=RESOLUTIONS,
    mode="eval_all",
    cache_features=CACHE_FEATURES
)
test_ds = MultiResCelebDFDataset(
    test_samples,
    image_size=IMAGE_SIZE_FOR_MODEL,
    resolutions=RESOLUTIONS,
    mode="eval_all",
    cache_features=CACHE_FEATURES
)

print("\nSample image checks")
for i in random.sample(range(len(train_samples)), min(5, len(train_samples))):
    item = train_samples[i]
    raw = Image.open(item["path"]).convert("RGB")
    chosen_res = random.choice(RESOLUTIONS)
    print(f"Path: {item['path']}")
    print(f"Original image size on disk: {raw.size}, label={item['label']}, sampled_train_res={chosen_res}")

sample = train_ds[0]
print("\nTensor sanity")
print("RGB tensor shape:        ", tuple(sample["rgb"].shape))
print("Frequency tensor shape:  ", tuple(sample["freq"].shape))
print("Quality feature shape:   ", tuple(sample["quality_features"].shape))
print("Quality features values: ", sample["quality_features"].tolist())
print("Label:                   ", sample["label"].item())
print("Resolution bucket:       ", sample["resolution"].item())

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
    loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

train_loader = DataLoader(
    train_ds,
    shuffle=True,
    **loader_kwargs
)
val_loader = DataLoader(
    val_ds,
    shuffle=False,
    **loader_kwargs
)
test_loader = DataLoader(
    test_ds,
    shuffle=False,
    **loader_kwargs
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = GatedDualBranchDeepfakeNet(
    branch_channels=BRANCH_CHANNELS,
    gate_features=6,
    num_classes=2
).to(DEVICE)

if DEVICE == "cuda":
    model = model.to(memory_format=torch.channels_last)

print("\nMODEL DEBUG")
print("model type:", type(model))
print("compiled?", "OptimizedModule" in str(type(model)) or hasattr(model, "_orig_mod"))

total_params, trainable_params = count_parameters(model)
print("\nModel summary")
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

flops_info = measure_flops(model, image_size=IMAGE_SIZE_FOR_MODEL, device=DEVICE)
if flops_info is not None:
    print("\nApprox FLOPs")
    for k, v in flops_info.items():
        if v is None:
            print(f"{k}: profiling failed")
        else:
            print(f"{k}: GFLOPs={v['flops'] / 1e9:.4f}, MACs={v['macs'] / 1e9:.4f}G")

print("\nTorch compile disabled. Using eager mode only.")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE == "cuda"))

history = {
    "train_loss": [],
    "train_acc": [],
    "train_auc": [],
    "val_loss": [],
    "val_acc": [],
    "val_auc": [],
    "epoch_time_sec": []
}

best_val_auc = -1.0
best_val_loss_proxy = float("inf")
total_train_start = time.time()

print("\nStarting training...")
for epoch in range(EPOCHS):
    epoch_start = time.time()
    print(f"\n{'='*25} Epoch {epoch+1}/{EPOCHS} {'='*25}")

    train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, DEVICE, epoch)
    val_metrics = evaluate_model(
        model,
        val_loader,
        DEVICE,
        split_name="Val",
        forced_route=None,
        threshold=UNCERTAIN_THRESHOLD
    )
    epoch_time = time.time() - epoch_start

    history["train_loss"].append(train_metrics["loss"])
    history["train_acc"].append(train_metrics["accuracy"])
    history["train_auc"].append(train_metrics["auc"])
    history["val_loss"].append(1.0 - val_metrics["auc"])
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_auc"].append(val_metrics["auc"])
    history["epoch_time_sec"].append(epoch_time)

    print("\nTrain")
    print(f"  Loss:      {train_metrics['loss']:.4f}")
    print(f"  Accuracy:  {train_metrics['accuracy']:.4f}")
    print(f"  AUC:       {train_metrics['auc']:.4f}")

    print("\nValidation")
    print(f"  Accuracy:  {val_metrics['accuracy']:.4f}")
    print(f"  AUC:       {val_metrics['auc']:.4f}")
    print(f"  Precision: {val_metrics['precision']:.4f}")
    print(f"  Recall:    {val_metrics['recall']:.4f}")
    print(f"  F1:        {val_metrics['f1']:.4f}")
    print(f"  Uncertain: {val_metrics['uncertain_rate_pct']:.2f}%")
    print(
        f"  Route use: fast={val_metrics['route_usage']['fast_exit1_pct']:.2f}% "
        f"medium={val_metrics['route_usage']['medium_exit2_pct']:.2f}% "
        f"full={val_metrics['route_usage']['full_exit3_pct']:.2f}%"
    )
    print(f"  Epoch time: {format_seconds(epoch_time)}")

    current_val_loss_proxy = 1.0 - val_metrics["auc"]
    if (val_metrics["auc"] > best_val_auc) or (
        abs(val_metrics["auc"] - best_val_auc) < 1e-8 and current_val_loss_proxy < best_val_loss_proxy
    ):
        best_val_auc = val_metrics["auc"]
        best_val_loss_proxy = current_val_loss_proxy
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Saved new best checkpoint.")

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

total_train_time = time.time() - total_train_start
print(f"\nTotal training time: {format_seconds(total_train_time)}")

plot_training_curves(history, CURVES_PNG)
with open(HISTORY_JSON, "w") as f:
    json.dump(history, f, indent=2)

print("\nLoading best checkpoint...")
state_dict = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)

print("\nRunning final evaluations...")
test_gated = evaluate_model(model, test_loader, DEVICE, split_name="Test Gated", forced_route=None, threshold=UNCERTAIN_THRESHOLD)
test_fast = evaluate_model(model, test_loader, DEVICE, split_name="Test Fast", forced_route=0, threshold=UNCERTAIN_THRESHOLD)
test_medium = evaluate_model(model, test_loader, DEVICE, split_name="Test Medium", forced_route=1, threshold=UNCERTAIN_THRESHOLD)
test_full = evaluate_model(model, test_loader, DEVICE, split_name="Test Full", forced_route=2, threshold=UNCERTAIN_THRESHOLD)

print_eval_summary("TEST GATED", test_gated)
print_eval_summary("TEST FAST ONLY", test_fast)
print_eval_summary("TEST MEDIUM ONLY", test_medium)
print_eval_summary("TEST FULL ONLY", test_full)

save_confusion_matrix(test_gated["confusion_matrix"], CONFUSION_MATRIX_PNG, title="Test Confusion Matrix - Gated")

inference_benchmarks = {
    "gated": measure_inference_time(model, test_loader, DEVICE, forced_route=None, threshold=UNCERTAIN_THRESHOLD, num_batches=20),
    "fast": measure_inference_time(model, test_loader, DEVICE, forced_route=0, threshold=UNCERTAIN_THRESHOLD, num_batches=20),
    "medium": measure_inference_time(model, test_loader, DEVICE, forced_route=1, threshold=UNCERTAIN_THRESHOLD, num_batches=20),
    "full": measure_inference_time(model, test_loader, DEVICE, forced_route=2, threshold=UNCERTAIN_THRESHOLD, num_batches=20),
}

print("\nInference benchmarks")
for name, bench in inference_benchmarks.items():
    print(f"  {name}: {bench['avg_image_time_ms']:.2f} ms/image, FPS={bench['fps']:.2f}")

checkpoint_size_mb = os.path.getsize(BEST_MODEL_PATH) / (1024 * 1024)

summary = {
    "config": {
        # "data_root": DATA_ROOT,  # CelebDF version
        "frames_dir": FRAMES_DIR,
        "metadata_csv": METADATA_CSV,
        "dataset_name": "FaceForensics++ C23",
        "resolutions": RESOLUTIONS,
        "image_size_for_model": IMAGE_SIZE_FOR_MODEL,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "uncertain_threshold": UNCERTAIN_THRESHOLD,
        "device": DEVICE,
        "num_workers": NUM_WORKERS,
        "cache_features": CACHE_FEATURES,
        "branch_channels": BRANCH_CHANNELS,
        "train_mode": "one random resolution per image per epoch",
        "eval_mode": "all requested resolutions",
        "use_torch_compile": USE_TORCH_COMPILE
    },
    "model_stats": {
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "checkpoint_size_mb": float(checkpoint_size_mb),
        "flops_info": flops_info
    },
    "timing": {
        "total_training_time_sec": float(total_train_time),
        "total_training_time_pretty": format_seconds(total_train_time)
    },
    "test_gated": test_gated,
    "test_fast": test_fast,
    "test_medium": test_medium,
    "test_full": test_full,
    "inference_benchmarks": inference_benchmarks
}

with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

observations = generate_observations(summary)
with open(OBSERVATIONS_TXT, "w") as f:
    f.write(observations)

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"Best checkpoint:          {BEST_MODEL_PATH}")
print(f"Checkpoint size:          {checkpoint_size_mb:.2f} MB")
print(f"Total params:             {total_params:,}")
print(f"Trainable params:         {trainable_params:,}")
print(f"Total training time:      {format_seconds(total_train_time)}")
print(f"Training curves saved:    {CURVES_PNG}")
print(f"Confusion matrix saved:   {CONFUSION_MATRIX_PNG}")
print(f"History JSON saved:       {HISTORY_JSON}")
print(f"Summary JSON saved:       {SUMMARY_JSON}")
print(f"Observations saved:       {OBSERVATIONS_TXT}")

print("\nKey test results")
print(f"Gated Accuracy:           {test_gated['accuracy']:.4f}")
print(f"Gated AUC:                {test_gated['auc']:.4f}")
print(f"Gated Uncertain rate:     {test_gated['uncertain_rate_pct']:.2f}%")
print(f"Fast route usage:         {test_gated['route_usage']['fast_exit1_pct']:.2f}%")
print(f"Medium route usage:       {test_gated['route_usage']['medium_exit2_pct']:.2f}%")
print(f"Full route usage:         {test_gated['route_usage']['full_exit3_pct']:.2f}%")

if flops_info is not None:
    print("\nApprox route GFLOPs")
    for name, info in flops_info.items():
        if info is None:
            print(f"  {name}: profiling failed")
        else:
            print(f"  {name}: {info['flops'] / 1e9:.4f} GFLOPs")

print("\nObservations preview")
print("-" * 70)
print(observations)
print("-" * 70)

  Using cached scikit_learn-1.8.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached matplotlib-3.10.9-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached pillow-12.2.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached thop-0.1.1.post2209072238-py3-none-any.whl.metadata (2.7 kB)
  Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp311-cp311-manyli

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.11.0+cu130
CUDA available: True
Device: cuda
THOP available: True
USE_TORCH_COMPILE: True
FRAMES_DIR: /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames
METADATA_CSV: /home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv

FINAL RESEARCH PIPELINE: FASTER GATED DUAL BRANCH DEEPFAKE DETECTOR

Loaded FF++ metadata
                                          frame_path  \
0  /home/jovyan/.cache/kagglehub/datasets/xdxd003...   
1  /home/jovyan/.cache/kagglehub/datasets/xdxd003...   
2  /home/jovyan/.cache/kagglehub/datasets/xdxd003...   
3  /home/jovyan/.cache/kagglehub/datasets/xdxd003...   
4  /home/jovyan/.cache/kagglehub/datasets/xdxd003...   

                                          frame_name  \
0  Train_1_FaceForensics++_C23__NeuralTextures__5...   
1  Train_1_FaceForensics++_C23__NeuralTextures__5...   
2  Train_1_FaceForensics++_C23__NeuralTextures__5...   
3  Train_1_FaceForensics+

Train Epoch 1:  11%|█▏        | 159/1399 [01:20<05:19,  3.88it/s, loss=0.6671, acc=0.8470]

!pip install opencv-python-headless matplotlib

In [ ]:
!ls /home/jovyan/.cache/kagglehub/datasets/pranabr0y/celebdf-v2image-dataset/versions/1/Celeb_V2

In [ ]:
print(test)

In [4]:
import numpy as np
import torch
import torchvision

print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

numpy: 2.4.3
torch: 2.11.0+cu130
torchvision: 0.26.0+cu130


In [3]:
!pip install scikit-learn torch torchvision numpy

  Using cached scikit_learn-1.8.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached torch-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached cuda_bindings-13.2.0-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
  Using cached nvidia_cudnn_cu13-

In [8]:
# FF++ 
!pip install tqdm scikit-learn matplotlib pillow opencv-python-headless thop pandas

import os, gc, cv2, json, time, random, copy, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    from thop import profile
    THOP_AVAILABLE = True
except Exception:
    THOP_AVAILABLE = False

# =========================
# Config
# =========================
FRAMES_DIR = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames"
METADATA_CSV = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv"
RESULTS_DIR = "./gated_research_results_ffpp"
os.makedirs(RESULTS_DIR, exist_ok=True)

RESOLUTIONS = [128, 224, 256, 384]
IMAGE_SIZE_FOR_MODEL = 224

BATCH_SIZE = 32
EPOCHS = 15
LR = 2e-4
WEIGHT_DECAY = 1e-4
SEED = 42
UNCERTAIN_THRESHOLD = 0.65
USE_AMP = True
USE_TORCH_COMPILE = False

# Important memory fixes
CACHE_FEATURES = False
NUM_WORKERS = 2
PIN_MEMORY = True
PERSISTENT_WORKERS = False
PREFETCH_FACTOR = 2

BRANCH_CHANNELS = (12, 20, 32, 48)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_gated_dual_branch_ffpp.pth")
HISTORY_JSON = os.path.join(RESULTS_DIR, "history_ffpp.json")
SUMMARY_JSON = os.path.join(RESULTS_DIR, "summary_ffpp.json")
OBSERVATIONS_TXT = os.path.join(RESULTS_DIR, "observations_ffpp.txt")
CONFUSION_MATRIX_PNG = os.path.join(RESULTS_DIR, "confusion_matrix_test_gated_ffpp.png")
CURVES_PNG = os.path.join(RESULTS_DIR, "training_curves_ffpp.png")

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)
print("THOP available:", THOP_AVAILABLE)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def format_seconds(sec):
    return f"{sec:.2f} sec" if sec < 60 else f"{sec/60:.2f} min"

def load_ffpp_metadata(metadata_csv):
    df = pd.read_csv(metadata_csv)
    required_cols = {"frame_path", "split", "label", "label_name", "video_id"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"metadata.csv missing columns: {missing}")
    df = df[df["frame_path"].apply(os.path.exists)].reset_index(drop=True)
    print("\nLoaded FF++ metadata")
    print(f"Total usable frames: {len(df)}")
    return df

def scan_ffpp_dataset(metadata_csv):
    df = load_ffpp_metadata(metadata_csv)
    print("\nScanning FF++ dataset structure...")
    for split in ["Train", "Val", "Test"]:
        split_df = df[df["split"].str.lower() == split.lower()]
        real_count = int((split_df["label"] == 0).sum())
        fake_count = int((split_df["label"] == 1).sum())
        print(f"{split}: real={real_count}, fake={fake_count}, total={len(split_df)}")
    return df

def build_base_samples_from_metadata(df, split):
    split_df = df[df["split"].str.lower() == split.lower()].reset_index(drop=True)
    return [
        {
            "path": row["frame_path"],
            "label": int(row["label"]),
            "split": row["split"],
            "video_id": row["video_id"],
        }
        for _, row in split_df.iterrows()
    ]

def print_dataset_sanity(train_samples, val_samples, test_samples):
    print("\nDataset sanity check")
    print(f"Train base images: {len(train_samples)}")
    print(f"Val base images:   {len(val_samples)}")
    print(f"Test base images:  {len(test_samples)}")
    print(f"Train effective samples per epoch: {len(train_samples)}")
    print(f"Val effective samples:             {len(val_samples) * len(RESOLUTIONS)}")
    print(f"Test effective samples:            {len(test_samples) * len(RESOLUTIONS)}")

def compute_blur_and_sharpness_from_rgb(np_rgb):
    gray = cv2.cvtColor(np_rgb, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    sharpness = float(np.var(lap))
    blur_proxy = 1.0 / (sharpness + 1e-6)
    return blur_proxy, sharpness

def compute_jpeg_compression_proxy(path):
    file_size_kb = os.path.getsize(path) / 1024.0
    ext = Path(path).suffix.lower()
    return float(1.0 / (file_size_kb + 1e-6)) if ext in [".jpg", ".jpeg"] else 0.0

def compute_dct_tensor(pil_img, out_size=224):
    gray = np.array(pil_img.convert("L")).astype(np.float32) / 255.0
    dct = cv2.dct(gray)
    dct = np.log1p(np.abs(dct))
    dct = (dct - dct.min()) / (dct.max() - dct.min() + 1e-6)
    dct = cv2.resize(dct, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    return torch.tensor(dct, dtype=torch.float32).unsqueeze(0)

def build_quality_features(pil_img, path, resized_resolution):
    np_rgb = np.array(pil_img.convert("RGB"))
    blur_proxy, sharpness = compute_blur_and_sharpness_from_rgb(np_rgb)
    compression_proxy = compute_jpeg_compression_proxy(path)

    feat = np.array(
        [
            resized_resolution / 224.0,
            min(blur_proxy * 100.0, 10.0),
            min(sharpness / 500.0, 10.0),
            min(compression_proxy * 50.0, 10.0),
            np.mean(np_rgb) / 255.0,
            np.std(np_rgb) / 255.0,
        ],
        dtype=np.float32,
    )
    return feat

def make_gate_targets_from_quality_features(qf_batch):
    targets = []
    qf_np = qf_batch.detach().cpu().numpy()
    for qf in qf_np:
        res_norm, blur_norm, sharp_norm, compression_norm, brightness, contrast = qf
        approx_res = res_norm * 224.0
        if approx_res >= 160 and sharp_norm >= 0.18 and compression_norm <= 0.8:
            targets.append(0)
        elif approx_res >= 128 and sharp_norm >= 0.06:
            targets.append(1)
        else:
            targets.append(2)
    return torch.tensor(targets, dtype=torch.long)

class MultiResCelebDFDataset(Dataset):
    def __init__(self, base_samples, image_size=224, resolutions=(128, 224, 256, 384), mode="train_random", cache_features=False):
        self.base_samples = base_samples
        self.image_size = image_size
        self.resolutions = list(resolutions)
        self.mode = mode
        self.cache_features = cache_features
        self.cache = {}

        if self.mode == "eval_all":
            self.samples = []
            for s in self.base_samples:
                for res in self.resolutions:
                    self.samples.append({**s, "resolution": res})
        else:
            self.samples = self.base_samples

        self.mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)

    def __len__(self):
        return len(self.samples)

    def pil_to_rgb_tensor(self, pil_img):
        arr = np.array(pil_img).astype(np.float32) / 255.0
        arr = np.transpose(arr, (2, 0, 1))
        tensor = torch.from_numpy(arr).float()
        return (tensor - self.mean) / self.std

    def _make_item(self, path, label, resolution):
        cache_key = (path, resolution)

        if self.cache_features and cache_key in self.cache:
            rgb, freq, quality_feat = self.cache[cache_key]
        else:
            with Image.open(path) as img:
                img = img.convert("RGB")
                img_res = img.resize((resolution, resolution), Image.BILINEAR)

            quality_feat_np = build_quality_features(img_res, path, resolution)
            img_for_model = img_res.resize((self.image_size, self.image_size), Image.BILINEAR)

            rgb = self.pil_to_rgb_tensor(img_for_model)
            freq = compute_dct_tensor(img_for_model, out_size=self.image_size)
            quality_feat = torch.tensor(quality_feat_np, dtype=torch.float32)

            if self.cache_features:
                self.cache[cache_key] = (rgb, freq, quality_feat)

        return {
            "rgb": rgb,
            "freq": freq,
            "quality_features": quality_feat,
            "label": torch.tensor(label, dtype=torch.long),
            "resolution": torch.tensor(resolution, dtype=torch.long),
            "path": path,
        }

    def __getitem__(self, idx):
        sample = self.samples[idx]
        resolution = random.choice(self.resolutions) if self.mode == "train_random" else sample["resolution"]
        return self._make_item(sample["path"], sample["label"], resolution)

class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=None, groups=1, act=True):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True) if act else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = ConvBNAct(in_ch, in_ch, kernel_size=3, stride=stride, groups=in_ch)
        self.pw = ConvBNAct(in_ch, out_ch, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        return self.pw(self.dw(x))

class ExitHead(nn.Module):
    def __init__(self, in_ch, hidden_dim=64, num_classes=2, dropout=0.2):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_ch, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        feat = self.pool(x).flatten(1)
        logits = self.fc(feat)
        return logits, feat

class AdaptiveGate(nn.Module):
    def __init__(self, in_features=6, hidden_dim=32, num_routes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, num_routes),
        )

    def forward(self, qf):
        logits = self.net(qf)
        probs = torch.softmax(logits, dim=1)
        route_idx = probs.argmax(dim=1)
        return logits, probs, route_idx

class EarlyExitBranch(nn.Module):
    def __init__(self, in_ch, channels=(12, 20, 32, 48), num_classes=2, exit_hidden=48):
        super().__init__()
        c1, c2, c3, c4 = channels

        self.stem = ConvBNAct(in_ch, c1, kernel_size=3, stride=2)
        self.stage1 = nn.Sequential(
            DepthwiseSeparableBlock(c1, c2, stride=2),
            DepthwiseSeparableBlock(c2, c2, stride=1),
        )
        self.stage2 = nn.Sequential(
            DepthwiseSeparableBlock(c2, c3, stride=2),
            DepthwiseSeparableBlock(c3, c3, stride=1),
        )
        self.stage3 = nn.Sequential(
            DepthwiseSeparableBlock(c3, c4, stride=2),
            DepthwiseSeparableBlock(c4, c4, stride=1),
        )

        self.exit1 = ExitHead(c2, hidden_dim=exit_hidden, num_classes=num_classes)
        self.exit2 = ExitHead(c3, hidden_dim=exit_hidden, num_classes=num_classes)
        self.exit3 = ExitHead(c4, hidden_dim=exit_hidden, num_classes=num_classes)

    def forward_all(self, x):
        x = self.stem(x)
        x1 = self.stage1(x)
        e1_logits, e1_feat = self.exit1(x1)
        x2 = self.stage2(x1)
        e2_logits, e2_feat = self.exit2(x2)
        x3 = self.stage3(x2)
        e3_logits, e3_feat = self.exit3(x3)
        return {
            "exit1_logits": e1_logits,
            "exit2_logits": e2_logits,
            "exit3_logits": e3_logits,
            "deep_feat": e3_feat,
        }

    def forward_route(self, x, route):
        x = self.stem(x)
        x1 = self.stage1(x)
        if route == 0:
            logits, feat = self.exit1(x1)
            return logits, feat

        x2 = self.stage2(x1)
        if route == 1:
            logits, feat = self.exit2(x2)
            return logits, feat

        x3 = self.stage3(x2)
        logits, feat = self.exit3(x3)
        return logits, feat

class GatedDualBranchDeepfakeNet(nn.Module):
    def __init__(self, branch_channels=(12, 20, 32, 48), gate_features=6, num_classes=2):
        super().__init__()
        self.spatial = EarlyExitBranch(3, channels=branch_channels, num_classes=num_classes, exit_hidden=48)
        self.frequency = EarlyExitBranch(1, channels=branch_channels, num_classes=num_classes, exit_hidden=48)
        self.gate = AdaptiveGate(in_features=gate_features, hidden_dim=32, num_routes=3)

        deep_dim = branch_channels[-1]
        self.fusion = nn.Sequential(
            nn.Linear(deep_dim * 2, 96),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(96, num_classes),
        )

    def forward(self, rgb, freq, qf):
        gate_logits, gate_probs, gate_routes = self.gate(qf)

        spatial_out = self.spatial.forward_all(rgb)
        freq_out = self.frequency.forward_all(freq)

        fused_feat = torch.cat([spatial_out["deep_feat"], freq_out["deep_feat"]], dim=1)
        fused_logits = self.fusion(fused_feat)

        return {
            "gate_logits": gate_logits,
            "gate_probs": gate_probs,
            "route_idx": gate_routes,
            "spatial_exit1": spatial_out["exit1_logits"],
            "spatial_exit2": spatial_out["exit2_logits"],
            "spatial_exit3": spatial_out["exit3_logits"],
            "freq_exit1": freq_out["exit1_logits"],
            "freq_exit2": freq_out["exit2_logits"],
            "freq_exit3": freq_out["exit3_logits"],
            "fused_logits": fused_logits,
        }

    def predict_fast_compute(self, rgb, freq, qf, threshold=0.65, forced_route=None):
        if forced_route is None:
            _, _, routes = self.gate(qf)
            # For mixed routes in a batch, use full forward for correctness.
            if len(torch.unique(routes)) > 1:
                return self.predict(rgb, freq, qf, threshold=threshold, forced_route=None)
            route = int(routes[0].item())
        else:
            route = int(forced_route)
            routes = torch.full((rgb.size(0),), route, dtype=torch.long, device=rgb.device)

        if route in [0, 1]:
            s_logits, _ = self.spatial.forward_route(rgb, route)
            f_logits, _ = self.frequency.forward_route(freq, route)
            final_logits = 0.5 * s_logits + 0.5 * f_logits
        else:
            s_logits, s_feat = self.spatial.forward_route(rgb, 2)
            f_logits, f_feat = self.frequency.forward_route(freq, 2)
            final_logits = self.fusion(torch.cat([s_feat, f_feat], dim=1))

        probs = torch.softmax(final_logits, dim=1)
        conf, pred = probs.max(dim=1)

        decision = []
        for c, p in zip(conf.tolist(), pred.tolist()):
            if c < threshold:
                decision.append("uncertain")
            elif p == 0:
                decision.append("real")
            else:
                decision.append("deepfake")

        return {
            "logits": final_logits,
            "probs": probs,
            "pred": pred,
            "conf": conf,
            "decision": decision,
            "routes": routes,
        }

    def predict(self, rgb, freq, qf, threshold=0.65, forced_route=None):
        out = self.forward(rgb, freq, qf)
        B = rgb.size(0)

        routes = out["route_idx"] if forced_route is None else torch.full((B,), forced_route, dtype=torch.long, device=rgb.device)
        final_logits = []

        for i in range(B):
            route = routes[i].item()
            if route == 0:
                logits_i = 0.5 * out["spatial_exit1"][i:i+1] + 0.5 * out["freq_exit1"][i:i+1]
            elif route == 1:
                logits_i = 0.5 * out["spatial_exit2"][i:i+1] + 0.5 * out["freq_exit2"][i:i+1]
            else:
                logits_i = out["fused_logits"][i:i+1]
            final_logits.append(logits_i)

        final_logits = torch.cat(final_logits, dim=0)
        probs = torch.softmax(final_logits, dim=1)
        conf, pred = probs.max(dim=1)

        decision = []
        for c, p in zip(conf.tolist(), pred.tolist()):
            if c < threshold:
                decision.append("uncertain")
            elif p == 0:
                decision.append("real")
            else:
                decision.append("deepfake")

        return {
            "logits": final_logits,
            "probs": probs,
            "pred": pred,
            "conf": conf,
            "decision": decision,
            "routes": routes,
            "raw_outputs": out,
        }

def compute_total_loss(outputs, labels, gate_targets, gate_weight=0.20):
    loss = 0.0
    loss += 0.08 * F.cross_entropy(outputs["spatial_exit1"], labels)
    loss += 0.12 * F.cross_entropy(outputs["spatial_exit2"], labels)
    loss += 0.15 * F.cross_entropy(outputs["spatial_exit3"], labels)
    loss += 0.08 * F.cross_entropy(outputs["freq_exit1"], labels)
    loss += 0.12 * F.cross_entropy(outputs["freq_exit2"], labels)
    loss += 0.15 * F.cross_entropy(outputs["freq_exit3"], labels)
    loss += 0.30 * F.cross_entropy(outputs["fused_logits"], labels)
    loss += gate_weight * F.cross_entropy(outputs["gate_logits"], gate_targets)
    return loss

def compute_classification_metrics(y_true, y_pred, y_score):
    acc = accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_score)
    except Exception:
        auc = 0.0

    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    return {
        "accuracy": float(acc),
        "auc": float(auc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "confusion_matrix": cm.tolist(),
    }

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

class RouteWrapper(nn.Module):
    def __init__(self, model, route):
        super().__init__()
        self.model = model
        self.route = route

    def forward(self, rgb, freq, qf):
        pred = self.model.predict_fast_compute(rgb, freq, qf, threshold=0.0, forced_route=self.route)
        return pred["logits"]

def remove_thop_buffers(module):
    for m in module.modules():
        for attr in ["total_ops", "total_params"]:
            if hasattr(m, attr):
                try:
                    delattr(m, attr)
                except Exception:
                    pass

def measure_flops(model, image_size=224, device="cpu"):
    if not THOP_AVAILABLE:
        return None

    rgb = torch.randn(1, 3, image_size, image_size).to(device)
    freq = torch.randn(1, 1, image_size, image_size).to(device)
    qf = torch.randn(1, 6).to(device)

    flops_info = {}
    for name, route in {"fast_route": 0, "medium_route": 1, "full_route": 2}.items():
        wrapper = RouteWrapper(copy.deepcopy(model), route).to(device).eval()
        remove_thop_buffers(wrapper)
        try:
            macs, params = profile(wrapper, inputs=(rgb, freq, qf), verbose=False)
            flops_info[name] = {
                "macs": float(macs),
                "flops": float(2 * macs),
                "params": float(params),
            }
        except Exception as e:
            print(f"THOP profiling failed for {name}: {e}")
            flops_info[name] = None
        finally:
            remove_thop_buffers(wrapper)
            del wrapper
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()

    return flops_info

def move_batch_to_device(batch, device, use_channels_last=False):
    rgb = batch["rgb"].to(device, non_blocking=True)
    freq = batch["freq"].to(device, non_blocking=True)
    qf = batch["quality_features"].to(device, non_blocking=True)
    labels = batch["label"].to(device, non_blocking=True)

    if use_channels_last:
        rgb = rgb.contiguous(memory_format=torch.channels_last)
        freq = freq.contiguous(memory_format=torch.channels_last)

    return rgb, freq, qf, labels

def train_one_epoch(model, loader, optimizer, scaler, device, epoch_idx):
    model.train()

    total_loss = 0.0
    total_count = 0
    running_y_true, running_y_pred, running_y_score = [], [], []

    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx+1}", leave=False)

    for batch in pbar:
        rgb, freq, qf, labels = move_batch_to_device(batch, device, use_channels_last=(device == "cuda"))
        gate_targets = make_gate_targets_from_quality_features(qf).to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(USE_AMP and device == "cuda")):
            outputs = model(rgb, freq, qf)
            loss = compute_total_loss(outputs, labels, gate_targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            probs = torch.softmax(outputs["fused_logits"], dim=1)[:, 1]
            preds = outputs["fused_logits"].argmax(dim=1)

        bs = rgb.size(0)
        total_loss += loss.item() * bs
        total_count += bs

        running_y_true.extend(labels.detach().cpu().numpy().tolist())
        running_y_pred.extend(preds.detach().cpu().numpy().tolist())
        running_y_score.extend(probs.detach().cpu().numpy().tolist())

        pbar.set_postfix({
            "loss": f"{total_loss / max(total_count, 1):.4f}",
            "acc": f"{accuracy_score(running_y_true, running_y_pred):.4f}",
        })

    metrics = compute_classification_metrics(running_y_true, running_y_pred, running_y_score)
    metrics["loss"] = float(total_loss / max(total_count, 1))
    return metrics

@torch.no_grad()
def evaluate_model(model, loader, device, split_name="Val", forced_route=None, threshold=0.65, fast_compute=False):
    model.eval()

    all_y_true, all_y_pred, all_y_score = [], [], []
    all_decisions, all_routes, all_resolutions = [], [], []

    start_time = time.time()
    pbar = tqdm(loader, desc=f"Eval {split_name}", leave=False)

    for batch in pbar:
        rgb = batch["rgb"].to(device, non_blocking=True)
        freq = batch["freq"].to(device, non_blocking=True)
        qf = batch["quality_features"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        resolutions = batch["resolution"].cpu().numpy().tolist()

        if device == "cuda":
            rgb = rgb.contiguous(memory_format=torch.channels_last)
            freq = freq.contiguous(memory_format=torch.channels_last)

        if fast_compute and forced_route is not None:
            pred_out = model.predict_fast_compute(rgb, freq, qf, threshold=threshold, forced_route=forced_route)
        else:
            pred_out = model.predict(rgb, freq, qf, threshold=threshold, forced_route=forced_route)

        all_y_true.extend(labels.detach().cpu().numpy().tolist())
        all_y_pred.extend(pred_out["pred"].detach().cpu().numpy().tolist())
        all_y_score.extend(pred_out["probs"][:, 1].detach().cpu().numpy().tolist())
        all_decisions.extend(pred_out["decision"])
        all_routes.extend(pred_out["routes"].detach().cpu().numpy().tolist())
        all_resolutions.extend(resolutions)

    elapsed = time.time() - start_time
    metrics = compute_classification_metrics(all_y_true, all_y_pred, all_y_score)

    route_counter = Counter(all_routes)
    metrics.update({
        "num_samples": int(len(all_y_true)),
        "uncertain_rate_pct": float(100.0 * sum(d == "uncertain" for d in all_decisions) / max(len(all_decisions), 1)),
        "route_usage": {
            "fast_exit1_pct": 100.0 * route_counter.get(0, 0) / max(len(all_routes), 1),
            "medium_exit2_pct": 100.0 * route_counter.get(1, 0) / max(len(all_routes), 1),
            "full_exit3_pct": 100.0 * route_counter.get(2, 0) / max(len(all_routes), 1),
        },
        "elapsed_sec": float(elapsed),
        "elapsed_pretty": format_seconds(elapsed),
    })

    per_resolution = {}
    for res in sorted(set(all_resolutions)):
        idxs = [i for i, r in enumerate(all_resolutions) if r == res]
        y_true_res = [all_y_true[i] for i in idxs]
        y_pred_res = [all_y_pred[i] for i in idxs]
        y_score_res = [all_y_score[i] for i in idxs]
        decisions_res = [all_decisions[i] for i in idxs]
        routes_res = [all_routes[i] for i in idxs]

        res_metrics = compute_classification_metrics(y_true_res, y_pred_res, y_score_res)
        rc = Counter(routes_res)
        res_metrics["uncertain_rate_pct"] = float(100.0 * sum(d == "uncertain" for d in decisions_res) / max(len(decisions_res), 1))
        res_metrics["route_usage"] = {
            "fast_exit1_pct": 100.0 * rc.get(0, 0) / max(len(routes_res), 1),
            "medium_exit2_pct": 100.0 * rc.get(1, 0) / max(len(routes_res), 1),
            "full_exit3_pct": 100.0 * rc.get(2, 0) / max(len(routes_res), 1),
        }
        res_metrics["num_samples"] = int(len(idxs))
        per_resolution[str(res)] = res_metrics

    metrics["per_resolution"] = per_resolution
    return metrics

def print_eval_summary(name, metrics):
    print(f"\n{name}")
    print(f"  Accuracy:        {metrics['accuracy']:.4f}")
    print(f"  AUC:             {metrics['auc']:.4f}")
    print(f"  Precision:       {metrics['precision']:.4f}")
    print(f"  Recall:          {metrics['recall']:.4f}")
    print(f"  F1:              {metrics['f1']:.4f}")
    print(f"  Uncertain rate:  {metrics['uncertain_rate_pct']:.2f}%")
    print(f"  Route usage: fast={metrics['route_usage']['fast_exit1_pct']:.2f}% medium={metrics['route_usage']['medium_exit2_pct']:.2f}% full={metrics['route_usage']['full_exit3_pct']:.2f}%")

def save_confusion_matrix(cm, path, title="Confusion Matrix"):
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["real", "fake"])
    plt.yticks(tick_marks, ["real", "fake"])
    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i][j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()

def plot_training_curves(history, path):
    epochs = list(range(1, len(history["train_loss"]) + 1))
    plt.figure(figsize=(14, 4))

    plt.subplot(1, 3, 1)
    plt.plot(epochs, history["train_loss"], marker="o", label="train_loss")
    plt.plot(epochs, history["val_loss"], marker="o", label="val_loss")
    plt.legend()
    plt.title("Loss")
    plt.xlabel("Epoch")

    plt.subplot(1, 3, 2)
    plt.plot(epochs, history["train_acc"], marker="o", label="train_acc")
    plt.plot(epochs, history["val_acc"], marker="o", label="val_acc")
    plt.legend()
    plt.title("Accuracy")
    plt.xlabel("Epoch")

    plt.subplot(1, 3, 3)
    plt.plot(epochs, history["val_auc"], marker="o", label="val_auc")
    plt.legend()
    plt.title("Validation AUC")
    plt.xlabel("Epoch")

    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()

@torch.no_grad()
def measure_inference_time(model, loader, device, forced_route=None, threshold=0.65, num_batches=20, fast_compute=True):
    model.eval()
    timings = []
    seen = 0

    for batch in loader:
        rgb = batch["rgb"].to(device, non_blocking=True)
        freq = batch["freq"].to(device, non_blocking=True)
        qf = batch["quality_features"].to(device, non_blocking=True)

        if device == "cuda":
            rgb = rgb.contiguous(memory_format=torch.channels_last)
            freq = freq.contiguous(memory_format=torch.channels_last)
            torch.cuda.synchronize()

        start = time.time()

        if fast_compute and forced_route is not None:
            _ = model.predict_fast_compute(rgb, freq, qf, threshold=threshold, forced_route=forced_route)
        else:
            _ = model.predict(rgb, freq, qf, threshold=threshold, forced_route=forced_route)

        if device == "cuda":
            torch.cuda.synchronize()

        timings.append(time.time() - start)
        seen += 1
        if seen >= num_batches:
            break

    avg_batch = float(np.mean(timings))
    avg_img = avg_batch / loader.batch_size
    fps = 1.0 / avg_img if avg_img > 0 else 0.0

    return {
        "avg_batch_time_sec": avg_batch,
        "avg_image_time_sec": float(avg_img),
        "avg_image_time_ms": float(avg_img * 1000.0),
        "fps": float(fps),
    }

def generate_observations(summary):
    tg = summary["test_gated"]
    tf = summary["test_fast"]
    tm = summary["test_medium"]
    tfull = summary["test_full"]

    lines = []
    lines.append("AUTOMATIC OBSERVATIONS")
    lines.append("=" * 60)
    lines.append(f"Gated test accuracy: {tg['accuracy']:.4f}")
    lines.append(f"Fast-only test accuracy: {tf['accuracy']:.4f}")
    lines.append(f"Medium-only test accuracy: {tm['accuracy']:.4f}")
    lines.append(f"Full-only test accuracy: {tfull['accuracy']:.4f}")
    lines.append("")
    lines.append("Route behavior")
    lines.append(
        f"Gate used Exit1 for {tg['route_usage']['fast_exit1_pct']:.2f}%, "
        f"Exit2 for {tg['route_usage']['medium_exit2_pct']:.2f}%, "
        f"and Exit3 for {tg['route_usage']['full_exit3_pct']:.2f}%."
    )
    return "\n".join(lines)

# =========================
# Main
# =========================
print("\n" + "=" * 70)
print("FINAL RESEARCH PIPELINE: SAFER GATED DUAL BRANCH DEEPFAKE DETECTOR")
print("=" * 70)

df_meta = scan_ffpp_dataset(METADATA_CSV)
train_samples = build_base_samples_from_metadata(df_meta, "Train")
val_samples = build_base_samples_from_metadata(df_meta, "Val")
test_samples = build_base_samples_from_metadata(df_meta, "Test")

print_dataset_sanity(train_samples, val_samples, test_samples)

train_ds = MultiResCelebDFDataset(train_samples, IMAGE_SIZE_FOR_MODEL, RESOLUTIONS, "train_random", CACHE_FEATURES)
val_ds = MultiResCelebDFDataset(val_samples, IMAGE_SIZE_FOR_MODEL, RESOLUTIONS, "eval_all", CACHE_FEATURES)
test_ds = MultiResCelebDFDataset(test_samples, IMAGE_SIZE_FOR_MODEL, RESOLUTIONS, "eval_all", CACHE_FEATURES)

sample = train_ds[0]
print("\nTensor sanity")
print("RGB tensor shape:", tuple(sample["rgb"].shape))
print("Frequency tensor shape:", tuple(sample["freq"].shape))
print("Quality feature shape:", tuple(sample["quality_features"].shape))
print("Label:", sample["label"].item())
print("Resolution bucket:", sample["resolution"].item())

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
    loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = GatedDualBranchDeepfakeNet(BRANCH_CHANNELS, gate_features=6, num_classes=2).to(DEVICE)

if DEVICE == "cuda":
    model = model.to(memory_format=torch.channels_last)

print("\nMODEL DEBUG")
print("model type:", type(model))
print("compiled?", "OptimizedModule" in str(type(model)) or hasattr(model, "_orig_mod"))

total_params, trainable_params = count_parameters(model)
print("\nModel summary")
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

flops_info = measure_flops(model, IMAGE_SIZE_FOR_MODEL, DEVICE)
if flops_info is not None:
    print("\nApprox FLOPs")
    for k, v in flops_info.items():
        if v is None:
            print(f"{k}: profiling failed")
        else:
            print(f"{k}: GFLOPs={v['flops'] / 1e9:.4f}, MACs={v['macs'] / 1e9:.4f}G")

print("\nTorch compile disabled. Cache disabled. Safer DataLoader active.")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE == "cuda"))

history = {
    "train_loss": [],
    "train_acc": [],
    "train_auc": [],
    "val_loss": [],
    "val_acc": [],
    "val_auc": [],
    "epoch_time_sec": [],
}

best_val_auc = -1.0
best_val_loss_proxy = float("inf")
total_train_start = time.time()

print("\nStarting training...")
for epoch in range(EPOCHS):
    epoch_start = time.time()
    print(f"\n{'='*25} Epoch {epoch+1}/{EPOCHS} {'='*25}")

    train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, DEVICE, epoch)
    val_metrics = evaluate_model(model, val_loader, DEVICE, "Val", forced_route=None, threshold=UNCERTAIN_THRESHOLD)

    epoch_time = time.time() - epoch_start

    history["train_loss"].append(train_metrics["loss"])
    history["train_acc"].append(train_metrics["accuracy"])
    history["train_auc"].append(train_metrics["auc"])
    history["val_loss"].append(1.0 - val_metrics["auc"])
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_auc"].append(val_metrics["auc"])
    history["epoch_time_sec"].append(epoch_time)

    print("\nTrain")
    print(f"  Loss:      {train_metrics['loss']:.4f}")
    print(f"  Accuracy:  {train_metrics['accuracy']:.4f}")
    print(f"  AUC:       {train_metrics['auc']:.4f}")

    print_eval_summary("Validation", val_metrics)
    print(f"  Epoch time: {format_seconds(epoch_time)}")

    current_val_loss_proxy = 1.0 - val_metrics["auc"]
    if (val_metrics["auc"] > best_val_auc) or (
        abs(val_metrics["auc"] - best_val_auc) < 1e-8 and current_val_loss_proxy < best_val_loss_proxy
    ):
        best_val_auc = val_metrics["auc"]
        best_val_loss_proxy = current_val_loss_proxy
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Saved new best checkpoint.")

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

total_train_time = time.time() - total_train_start
print(f"\nTotal training time: {format_seconds(total_train_time)}")

plot_training_curves(history, CURVES_PNG)
with open(HISTORY_JSON, "w") as f:
    json.dump(history, f, indent=2)

print("\nLoading best checkpoint...")
state_dict = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)

print("\nRunning final evaluations...")
test_gated = evaluate_model(model, test_loader, DEVICE, "Test Gated", forced_route=None, threshold=UNCERTAIN_THRESHOLD)
test_fast = evaluate_model(model, test_loader, DEVICE, "Test Fast", forced_route=0, threshold=UNCERTAIN_THRESHOLD, fast_compute=True)
test_medium = evaluate_model(model, test_loader, DEVICE, "Test Medium", forced_route=1, threshold=UNCERTAIN_THRESHOLD, fast_compute=True)
test_full = evaluate_model(model, test_loader, DEVICE, "Test Full", forced_route=2, threshold=UNCERTAIN_THRESHOLD, fast_compute=True)

print_eval_summary("TEST GATED", test_gated)
print_eval_summary("TEST FAST ONLY", test_fast)
print_eval_summary("TEST MEDIUM ONLY", test_medium)
print_eval_summary("TEST FULL ONLY", test_full)

save_confusion_matrix(test_gated["confusion_matrix"], CONFUSION_MATRIX_PNG, title="Test Confusion Matrix - Gated")

inference_benchmarks = {
    "gated": measure_inference_time(model, test_loader, DEVICE, forced_route=None, threshold=UNCERTAIN_THRESHOLD, num_batches=20, fast_compute=False),
    "fast": measure_inference_time(model, test_loader, DEVICE, forced_route=0, threshold=UNCERTAIN_THRESHOLD, num_batches=20, fast_compute=True),
    "medium": measure_inference_time(model, test_loader, DEVICE, forced_route=1, threshold=UNCERTAIN_THRESHOLD, num_batches=20, fast_compute=True),
    "full": measure_inference_time(model, test_loader, DEVICE, forced_route=2, threshold=UNCERTAIN_THRESHOLD, num_batches=20, fast_compute=True),
}

print("\nInference benchmarks")
for name, bench in inference_benchmarks.items():
    print(f"  {name}: {bench['avg_image_time_ms']:.2f} ms/image, FPS={bench['fps']:.2f}")

checkpoint_size_mb = os.path.getsize(BEST_MODEL_PATH) / (1024 * 1024)

summary = {
    "config": {
        "frames_dir": FRAMES_DIR,
        "metadata_csv": METADATA_CSV,
        "dataset_name": "FaceForensics++ C23",
        "resolutions": RESOLUTIONS,
        "image_size_for_model": IMAGE_SIZE_FOR_MODEL,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "uncertain_threshold": UNCERTAIN_THRESHOLD,
        "device": DEVICE,
        "num_workers": NUM_WORKERS,
        "cache_features": CACHE_FEATURES,
        "branch_channels": BRANCH_CHANNELS,
        "use_torch_compile": USE_TORCH_COMPILE,
    },
    "model_stats": {
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "checkpoint_size_mb": float(checkpoint_size_mb),
        "flops_info": flops_info,
    },
    "timing": {
        "total_training_time_sec": float(total_train_time),
        "total_training_time_pretty": format_seconds(total_train_time),
    },
    "test_gated": test_gated,
    "test_fast": test_fast,
    "test_medium": test_medium,
    "test_full": test_full,
    "inference_benchmarks": inference_benchmarks,
}

with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

observations = generate_observations(summary)
with open(OBSERVATIONS_TXT, "w") as f:
    f.write(observations)

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"Best checkpoint:          {BEST_MODEL_PATH}")
print(f"Checkpoint size:          {checkpoint_size_mb:.2f} MB")
print(f"Total params:             {total_params:,}")
print(f"Trainable params:         {trainable_params:,}")
print(f"Total training time:      {format_seconds(total_train_time)}")
print(f"Training curves saved:    {CURVES_PNG}")
print(f"Confusion matrix saved:   {CONFUSION_MATRIX_PNG}")
print(f"History JSON saved:       {HISTORY_JSON}")
print(f"Summary JSON saved:       {SUMMARY_JSON}")
print(f"Observations saved:       {OBSERVATIONS_TXT}")

print("\nKey test results")
print(f"Gated Accuracy:           {test_gated['accuracy']:.4f}")
print(f"Gated AUC:                {test_gated['auc']:.4f}")
print(f"Gated Uncertain rate:     {test_gated['uncertain_rate_pct']:.2f}%")
print(f"Fast route usage:         {test_gated['route_usage']['fast_exit1_pct']:.2f}%")
print(f"Medium route usage:       {test_gated['route_usage']['medium_exit2_pct']:.2f}%")
print(f"Full route usage:         {test_gated['route_usage']['full_exit3_pct']:.2f}%")

if flops_info is not None:
    print("\nApprox route GFLOPs")
    for name, info in flops_info.items():
        if info is None:
            print(f"  {name}: profiling failed")
        else:
            print(f"  {name}: {info['flops'] / 1e9:.4f} GFLOPs")

print("\nObservations preview")
print("-" * 70)
print(observations)
print("-" * 70)

Torch version: 2.11.0+cu130
CUDA available: True
Device: cuda
THOP available: True

FINAL RESEARCH PIPELINE: SAFER GATED DUAL BRANCH DEEPFAKE DETECTOR

Loaded FF++ metadata
Total usable frames: 55953

Scanning FF++ dataset structure...
Train: real=6400, fake=38356, total=44756
Val: real=800, fake=4800, total=5600
Test: real=800, fake=4797, total=5597

Dataset sanity check
Train base images: 44756
Val base images:   5600
Test base images:  5597
Train effective samples per epoch: 44756
Val effective samples:             22400
Test effective samples:            22388

Tensor sanity
RGB tensor shape: (3, 224, 224)
Frequency tensor shape: (1, 224, 224)
Quality feature shape: (6,)
Label: 1
Resolution bucket: 128

MODEL DEBUG
model type: <class '__main__.GatedDualBranchDeepfakeNet'>
compiled? False

Model summary
Total params:     38,537
Trainable params: 38,537

Approx FLOPs
fast_route: GFLOPs=0.0288, MACs=0.0144G
medium_route: GFLOPs=0.0367, MACs=0.0184G
full_route: GFLOPs=0.0408, MACs=0.02


Train
  Loss:      0.5176
  Accuracy:  0.8556
  AUC:       0.5482

Validation
  Accuracy:        0.8571
  AUC:             0.5063
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9231
  Uncertain rate:  0.00%
  Route usage: fast=67.08% medium=32.92% full=0.00%
  Epoch time: 27.73 min
Saved new best checkpoint.

========================= Epoch 2/15 =========================



Train
  Loss:      0.4436
  Accuracy:  0.8570
  AUC:       0.5897

Validation
  Accuracy:        0.8571
  AUC:             0.5145
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9231
  Uncertain rate:  0.00%
  Route usage: fast=65.75% medium=34.25% full=0.00%
  Epoch time: 24.39 min
Saved new best checkpoint.

========================= Epoch 3/15 =========================



Train
  Loss:      0.4314
  Accuracy:  0.8570
  AUC:       0.5943

Validation
  Accuracy:        0.8571
  AUC:             0.5177
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9231
  Uncertain rate:  0.00%
  Route usage: fast=66.68% medium=33.32% full=0.00%
  Epoch time: 25.21 min
Saved new best checkpoint.

========================= Epoch 4/15 =========================



Train
  Loss:      0.4235
  Accuracy:  0.8570
  AUC:       0.6097

Validation
  Accuracy:        0.8571
  AUC:             0.5242
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9231
  Uncertain rate:  0.00%
  Route usage: fast=64.83% medium=35.13% full=0.04%
  Epoch time: 24.26 min
Saved new best checkpoint.

========================= Epoch 5/15 =========================


...


Train
  Loss:      0.4012
  Accuracy:  0.8570
  AUC:       0.6571

Validation
  Accuracy:        0.8571
  AUC:             0.5154
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9231
  Uncertain rate:  0.00%
  Route usage: fast=65.80% medium=34.09% full=0.10%
  Epoch time: 21.37 min

========================= Epoch 13/15 =========================



Train
  Loss:      0.3991
  Accuracy:  0.8570
  AUC:       0.6687

Validation
  Accuracy:        0.8571
  AUC:             0.5244
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9231
  Uncertain rate:  0.00%
  Route usage: fast=66.04% medium=33.85% full=0.11%
  Epoch time: 21.03 min

========================= Epoch 14/15 =========================



Train
  Loss:      0.3973
  Accuracy:  0.8570
  AUC:       0.6751

Validation
  Accuracy:        0.8571
  AUC:             0.5153
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9231
  Uncertain rate:  0.00%
  Route usage: fast=64.53% medium=35.37% full=0.11%
  Epoch time: 21.52 min

========================= Epoch 15/15 =========================



Train
  Loss:      0.3967
  Accuracy:  0.8570
  AUC:       0.6776

Validation
  Accuracy:        0.8571
  AUC:             0.5127
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9231
  Uncertain rate:  0.00%
  Route usage: fast=64.86% medium=35.04% full=0.10%
  Epoch time: 22.49 min

Total training time: 334.93 min

Loading best checkpoint...

Running final evaluations...



TEST GATED
  Accuracy:        0.8571
  AUC:             0.5163
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9230
  Uncertain rate:  0.00%
  Route usage: fast=66.95% medium=32.97% full=0.08%

TEST FAST ONLY
  Accuracy:        0.8571
  AUC:             0.5295
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9230
  Uncertain rate:  0.00%
  Route usage: fast=100.00% medium=0.00% full=0.00%

TEST MEDIUM ONLY
  Accuracy:        0.8571
  AUC:             0.5101
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9230
  Uncertain rate:  0.00%
  Route usage: fast=0.00% medium=100.00% full=0.00%

TEST FULL ONLY
  Accuracy:        0.8571
  AUC:             0.4182
  Precision:       0.8571
  Recall:          1.0000
  F1:              0.9230
  Uncertain rate:  0.00%
  Route usage: fast=0.00% medium=0.00% full=100.00%

Inference benchmarks
  gated: 0.25 ms/image, FPS=4036.02
  fast: 0.05 ms/image, FPS=20298.34
  medium: 0.15 m

In [9]:
import os
import pandas as pd

METADATA_CSV = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv"

df = pd.read_csv(METADATA_CSV)

df["exists"] = df["frame_path"].apply(os.path.exists)
df = df[df["exists"]].reset_index(drop=True)

print("=" * 70)
print("FRAMES AND VIDEOS PER CLASS")
print("=" * 70)

overall = (
    df.groupby(["label", "label_name"])
    .agg(
        num_frames=("frame_path", "count"),
        num_videos=("video_id", "nunique")
    )
    .reset_index()
)

print("\nOverall:")
print(overall.to_string(index=False))

print("\n" + "=" * 70)
print("FRAMES AND VIDEOS PER CLASS BY SPLIT")
print("=" * 70)

by_split = (
    df.groupby(["split", "label", "label_name"])
    .agg(
        num_frames=("frame_path", "count"),
        num_videos=("video_id", "nunique")
    )
    .reset_index()
    .sort_values(["split", "label"])
)

print(by_split.to_string(index=False))

print("\n" + "=" * 70)
print("TOTAL FRAMES AND VIDEOS BY SPLIT")
print("=" * 70)

split_totals = (
    df.groupby("split")
    .agg(
        num_frames=("frame_path", "count"),
        num_videos=("video_id", "nunique")
    )
    .reset_index()
)

print(split_totals.to_string(index=False))

FRAMES AND VIDEOS PER CLASS

Overall:
 label label_name  num_frames  num_videos
     0       Real        8000        1000
     1       Fake       47953        6000

FRAMES AND VIDEOS PER CLASS BY SPLIT
split  label label_name  num_frames  num_videos
 Test      0       Real         800         100
 Test      1       Fake        4797         600
Train      0       Real        6400         800
Train      1       Fake       38356        4800
  Val      0       Real         800         100
  Val      1       Fake        4800         600

TOTAL FRAMES AND VIDEOS BY SPLIT
split  num_frames  num_videos
 Test        5597         700
Train       44756        5600
  Val        5600         700


In [10]:
import json

SUMMARY_JSON = "./gated_research_results_ffpp/summary_ffpp.json"

with open(SUMMARY_JSON, "r") as f:
    summary = json.load(f)

per_res = summary["test_gated"]["per_resolution"]

print("\nAccuracy based on resolution")
print("-" * 80)
print(f"{'Resolution':<12}{'Accuracy':<12}{'AUC':<10}{'Fast (%)':<12}{'Medium (%)':<14}{'Full (%)':<10}")
print("-" * 80)

for res in sorted(per_res.keys(), key=lambda x: int(x)):
    m = per_res[res]
    
    acc = m["accuracy"]
    auc = m["auc"]
    fast = m["route_usage"]["fast_exit1_pct"]
    medium = m["route_usage"]["medium_exit2_pct"]
    full = m["route_usage"]["full_exit3_pct"]

    print(f"{res:<12}{acc:<12.4f}{auc:<10.4f}{fast:<12.2f}{medium:<14.2f}{full:<10.2f}")


Accuracy based on resolution
--------------------------------------------------------------------------------
Resolution  Accuracy    AUC       Fast (%)    Medium (%)    Full (%)  
--------------------------------------------------------------------------------
128         0.8571      0.5043    0.00        100.00        0.00      
224         0.8571      0.5253    88.64       11.36         0.00      
256         0.8571      0.5214    89.78       10.22         0.00      
384         0.8571      0.5202    89.37       10.29         0.34      
